# DIGITAL LENDING: PORTFOLIO OPTIMIZATION
## MODULE 6 — Dynamic Risk-Based Pricing & Loan Structuring Optimization Engine
### PART 1 of 2 — Core Pricing Engine, Profitability Modeling, Segment & Behavioral Pricing

**Depends on**: Module 1–5 outputs  
**Primary Input**: `lending_features/master_feature_table.csv` + Module 5 risk scores  
**Outputs**: `pricing_engine/` — models, reports, dashboards, policies

---
**Part 1 covers:**
- Cells 1–2   : Imports, config, paths, helpers
- Cell  3     : Pricing Strategy Design
- Cell  4     : Data Loading & Module 5 integration
- Cell  5     : Business Rule Framework & Policy Matrix
- Cells 6–7   : Risk-Based Pricing Engine + Portfolio application
- Cell  8     : Tenure & Credit Limit Optimization
- Cell  9     : Full Profitability Modeling & RAROC
- Cell  10    : Segment-Based & Behavioral Pricing Intelligence
- Cell  11    : Approval-Pricing Integration
---

In [11]:
# =============================================================================
# CELL 1 — Install dependencies (run once)
# =============================================================================
!pip install pandas numpy scipy scikit-learn xgboost lightgbm optuna \
           matplotlib seaborn plotly shap joblib statsmodels --quiet


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: C:\Users\abhir\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [12]:
# =============================================================================
# CELL 2 — Imports & Global Configuration
# =============================================================================
import os, sys, json, warnings, logging, joblib, time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize_scalar
from scipy.stats import f_oneway

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

try:
    import shap
    SHAP_OK = True
except ImportError:
    SHAP_OK = False

warnings.filterwarnings("ignore")

try:
    get_ipython()
    matplotlib.use("inline")
except NameError:
    matplotlib.use("Agg")

SEED = 42
np.random.seed(SEED)

# ── Palette ───────────────────────────────────────────────────────────────
PAL = {
    "primary":   "#2C5F8A",
    "danger":    "#D94F3D",
    "success":   "#2E8B57",
    "warning":   "#E8A838",
    "neutral":   "#6B7280",
    "highlight": "#7B2D8B",
    "bg":        "#F8F9FA",
    "gold":      "#C9A227",
}
GRADE_COLORS = {
    "A": "#2E8B57", "B": "#7DCE82",
    "C": "#E8A838", "D": "#E07B39", "E": "#D94F3D",
}

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({
    "figure.facecolor": PAL["bg"],
    "axes.facecolor":   "white",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.titleweight": "bold",
    "axes.titlesize":   12,
})

# ── Paths ─────────────────────────────────────────────────────────────────
BASE_DIR  = r"C:\Users\abhir\OneDrive\Desktop\iitg"
FEAT_DIR  = os.path.join(BASE_DIR, "lending_features")
PE_DIR    = os.path.join(BASE_DIR, "pricing_engine")
MDL_DIR   = os.path.join(PE_DIR,  "models")
RPT_DIR   = os.path.join(PE_DIR,  "reports")
SIM_DIR   = os.path.join(PE_DIR,  "simulations")
EXP_DIR   = os.path.join(PE_DIR,  "explainability")
DASH_DIR  = os.path.join(PE_DIR,  "dashboards")
PIP_DIR   = os.path.join(PE_DIR,  "pipelines")
POL_DIR   = os.path.join(PE_DIR,  "policies")
NB_DIR    = os.path.join(PE_DIR,  "notebooks")

for d in [PE_DIR,MDL_DIR,RPT_DIR,SIM_DIR,EXP_DIR,DASH_DIR,PIP_DIR,POL_DIR,NB_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

# ── Logging ───────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.FileHandler(os.path.join(PE_DIR, "module6.log"), mode="w", encoding="utf-8"),
        logging.StreamHandler(sys.stdout),
    ],
)
log = logging.getLogger("PricingEngine")
log.info("Module 6 Part 1 — Dynamic Pricing pipeline started.")

# ── Helpers ───────────────────────────────────────────────────────────────
def savefig(name, dpi=150):
    path = os.path.join(DASH_DIR, name)
    plt.savefig(path, dpi=dpi, bbox_inches="tight", facecolor=PAL["bg"])
    plt.close()
    log.info("  Figure: %s", name)
    return path

def section(title):
    bar = "═" * 70
    print(f"\n{bar}\n  {title}\n{bar}")

def get_col(df, col, default):
    """Safe column extraction with default fallback."""
    return df[col].fillna(default).values if col in df.columns else np.full(len(df), default)

print("✅  Module 6 Part 1 — Configuration loaded. All pricing_engine/ dirs ready.")

14:26:30 | INFO     | Module 6 Part 1 — Dynamic Pricing pipeline started.
✅  Module 6 Part 1 — Configuration loaded. All pricing_engine/ dirs ready.


In [13]:
# =============================================================================
# CELL 3 — STEP 1: Pricing Strategy Design
# =============================================================================

section("STEP 1 — PRICING STRATEGY DESIGN")

STRATEGY_DOC = """
╔══════════════════════════════════════════════════════════════════════╗
║  MODULE 6 — DYNAMIC RISK-BASED PRICING STRATEGY                      ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  BUSINESS OBJECTIVES OF PRICING:                                     ║
║  1. Cover cost of funds + operational costs + risk premium           ║
║  2. Maximise risk-adjusted net interest margin (NIM)                 ║
║  3. Price borrowers proportional to actual default probability       ║
║  4. Remain competitive to attract prime borrowers                    ║
║  5. Prevent adverse selection (under-pricing = higher losses)        ║
║                                                                      ║
║  RATE CONSTRUCTION FORMULA:                                          ║
║  Rate = CoF + Op_Spread + Risk_Premium + Margin + Beh_Modifier      ║
║  Risk_Premium = f(PD, LGD, EAD, Tenure)                             ║
║  Higher PD → Higher risk premium to cover expected loss             ║
║                                                                      ║
║  GROWTH vs PROFITABILITY TRADEOFF:                                   ║
║  Aggressive pricing → lower rates → more volume → higher risk       ║
║  Conservative pricing → higher rates → less volume → quality ↑      ║
║  Optimal = CLV-maximising rate balancing volume and margin           ║
║                                                                      ║
║  TENURE IMPACT ON EXPECTED LOSS:                                     ║
║  EL(T) = [1-(1-PD)^(T/12)] × LGD × EAD                            ║
║  Longer tenure → higher cumulative PD → higher EL                   ║
║  Short tenure = lower EL but lower total interest income             ║
║                                                                      ║
║  PRICING COMPONENTS:                                                 ║
║  Base Rate        : 8.50%  (cost of funds proxy)                    ║
║  Operational Spread: 2.00% (fixed ops + servicing)                  ║
║  Risk Premium     : 0–18%  (PD-driven, grade-differentiated)        ║
║  Target Margin    : 2.00%  (NIM target)                             ║
║  Behavioral Mod   : ±2%    (volatility / engagement signal)         ║
║  Segment Override : ±4%    (persona / channel adjustment)           ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(STRATEGY_DOC)
with open(os.path.join(POL_DIR, "pricing_strategy.txt"), "w", encoding="utf-8") as f:
    f.write(STRATEGY_DOC)

# ── Core pricing constants ─────────────────────────────────────────────────
PRICING_CONFIG = {
    "base_rate":           8.50,
    "operational_spread":  2.00,
    "target_margin":       2.00,
    "lgd_unsecured":       0.60,
    "lgd_secured":         0.30,
    "min_rate":           10.00,
    "max_rate":           36.00,
    "min_tenure_months":   3,
    "max_tenure_months":  84,
    "emi_burden_ceiling":  0.45,
    "max_lti":             5.0,
}

with open(os.path.join(POL_DIR, "pricing_config.json"), "w") as f:
    json.dump(PRICING_CONFIG, f, indent=2)
log.info("Pricing strategy and config saved.")
print("\n  ✅  Pricing config saved to policies/pricing_config.json")


══════════════════════════════════════════════════════════════════════
  STEP 1 — PRICING STRATEGY DESIGN
══════════════════════════════════════════════════════════════════════

╔══════════════════════════════════════════════════════════════════════╗
║  MODULE 6 — DYNAMIC RISK-BASED PRICING STRATEGY                      ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  BUSINESS OBJECTIVES OF PRICING:                                     ║
║  1. Cover cost of funds + operational costs + risk premium           ║
║  2. Maximise risk-adjusted net interest margin (NIM)                 ║
║  3. Price borrowers proportional to actual default probability       ║
║  4. Remain competitive to attract prime borrowers                    ║
║  5. Prevent adverse selection (under-pricing = higher losses)        ║
║                                                                      ║
║  RATE CONSTRUCTI

In [14]:
# =============================================================================
# CELL 4 — Data Loading & Module 5 Integration
# =============================================================================

section("DATA LOADING & MODULE 5 INTEGRATION")

path = os.path.join(FEAT_DIR, "master_feature_table.csv")
if not os.path.exists(path):
    raise FileNotFoundError(f"master_feature_table.csv not found at {path}. Run Module 2 first.")

df_raw = pd.read_csv(path, low_memory=False)
log.info("Raw data: %s rows × %s cols", f"{len(df_raw):,}", df_raw.shape[1])
df = df_raw.copy()

# ── Load Module 5 scores if available ────────────────────────────────────
m5_score_path = os.path.join(BASE_DIR, "risk_models", "metrics", "scored_test_set.csv")
if os.path.exists(m5_score_path):
    m5_scores = pd.read_csv(m5_score_path)
    if "loan_id" in df.columns and "loan_id" in m5_scores.columns:
        merge_cols = [c for c in ["loan_id","pd_score","risk_score","risk_grade"]
                      if c in m5_scores.columns]
        df = df.merge(m5_scores[merge_cols], on="loan_id", how="left")
    log.info("Module 5 scores merged.")
    print(f"  ✅  Module 5 scores loaded: {len(m5_scores):,} rows")
else:
    log.warning("Module 5 scores not found — PD synthesised from credit_score.")
    print("  ⚠  Module 5 scores not found — PD estimated from credit_score.")

# ── Ensure pd_score exists ────────────────────────────────────────────────
if "pd_score" not in df.columns:
    cs = df["credit_score"].fillna(650) if "credit_score" in df.columns else pd.Series([650]*len(df))
    df["pd_score"] = np.clip(1 - (cs - 300) / 600, 0.02, 0.80)

# ── Ensure risk_grade exists ──────────────────────────────────────────────
if "risk_grade" not in df.columns:
    def _grade(pd_val):
        if pd_val < 0.05:  return "A"
        if pd_val < 0.10:  return "B"
        if pd_val < 0.18:  return "C"
        if pd_val < 0.30:  return "D"
        return "E"
    df["risk_grade"] = df["pd_score"].apply(_grade)

# Clean grade to single letter (handles 'A — Prime' format from Module 5)
df["risk_grade_clean"] = df["risk_grade"].astype(str).str[0].str.upper()
df["risk_grade_clean"]  = df["risk_grade_clean"].where(
    df["risk_grade_clean"].isin(["A","B","C","D","E"]), other="C"
)

print(f"\n  Dataset shape        : {df.shape}")
print(f"  PD score range       : {df['pd_score'].min():.4f} – {df['pd_score'].max():.4f}")
print(f"  Avg PD               : {df['pd_score'].mean():.4f}")
print(f"  Risk grade distribution:")
print(df["risk_grade_clean"].value_counts().sort_index().to_string())


══════════════════════════════════════════════════════════════════════
  DATA LOADING & MODULE 5 INTEGRATION
══════════════════════════════════════════════════════════════════════
14:26:34 | INFO     | Raw data: 65,000 rows × 76 cols
14:26:34 | INFO     | Module 5 scores merged.
  ✅  Module 5 scores loaded: 4,917 rows

  Dataset shape        : (65000, 79)
  PD score range       : 0.0533 – 0.8000
  Avg PD               : 0.5765
  Risk grade distribution:
risk_grade_clean
B      168
D       27
E    64805


In [15]:
# =============================================================================
# CELL 5 — STEP 2: Business Rule Framework & Pricing Policy Matrix
# =============================================================================

section("STEP 2 — BUSINESS RULE FRAMEWORK & PRICING POLICY MATRIX")

PRICING_POLICY = {
    "A": {
        "label": "Prime",
        "rate_floor": 10.0, "rate_ceil": 14.0,
        "tenure_min": 6,    "tenure_max": 60,
        "max_lti": 5.0,     "lgd": 0.50,
        "risk_premium_base": 0.5,
        "approval": "Auto-Approve",
        "limit_multiplier": 3.0,  # × monthly income
        "behavioral_mod_cap": 1.0,
    },
    "B": {
        "label": "Near-Prime",
        "rate_floor": 14.0, "rate_ceil": 19.0,
        "tenure_min": 6,    "tenure_max": 48,
        "max_lti": 4.0,     "lgd": 0.55,
        "risk_premium_base": 3.5,
        "approval": "Auto-Approve",
        "limit_multiplier": 2.0,
        "behavioral_mod_cap": 1.5,
    },
    "C": {
        "label": "Sub-Prime",
        "rate_floor": 19.0, "rate_ceil": 26.0,
        "tenure_min": 3,    "tenure_max": 36,
        "max_lti": 3.0,     "lgd": 0.60,
        "risk_premium_base": 8.0,
        "approval": "Conditional",
        "limit_multiplier": 1.5,
        "behavioral_mod_cap": 2.0,
    },
    "D": {
        "label": "High Risk",
        "rate_floor": 26.0, "rate_ceil": 32.0,
        "tenure_min": 3,    "tenure_max": 24,
        "max_lti": 2.0,     "lgd": 0.65,
        "risk_premium_base": 14.0,
        "approval": "Manual Review",
        "limit_multiplier": 1.0,
        "behavioral_mod_cap": 2.5,
    },
    "E": {
        "label": "Very High Risk",
        "rate_floor": 32.0, "rate_ceil": 36.0,
        "tenure_min": 3,    "tenure_max": 12,
        "max_lti": 1.0,     "lgd": 0.70,
        "risk_premium_base": 18.0,
        "approval": "Decline",
        "limit_multiplier": 0.5,
        "behavioral_mod_cap": 3.0,
    },
}

# Segment-specific pricing overrides (from Module 4 personas)
SEGMENT_PRICING_OVERRIDE = {
    "Prime Stable Borrowers":         {"rate_adj": -1.0, "tenure_adj": +12, "limit_adj": +1.2},
    "High-Value Loyal Customers":     {"rate_adj": -1.5, "tenure_adj": +24, "limit_adj": +1.3},
    "Digitally Engaged Mid-Tier":     {"rate_adj": -0.5, "tenure_adj":   0, "limit_adj": +1.0},
    "Volatile Gig Economy Borrowers": {"rate_adj": +2.0, "tenure_adj": -12, "limit_adj":  0.8},
    "High-Risk Financially Stressed": {"rate_adj": +3.0, "tenure_adj": -18, "limit_adj":  0.6},
    "Subprime Over-Leveraged":        {"rate_adj": +4.0, "tenure_adj": -24, "limit_adj":  0.5},
    "Emerging Growth Segment":        {"rate_adj":  0.0, "tenure_adj":   0, "limit_adj":  1.0},
    "Low-Profit Value Drainers":      {"rate_adj": +2.5, "tenure_adj": -12, "limit_adj":  0.7},
}

# Save
with open(os.path.join(POL_DIR, "pricing_policy_matrix.json"), "w") as f:
    json.dump(PRICING_POLICY, f, indent=2)
with open(os.path.join(POL_DIR, "segment_pricing_override.json"), "w") as f:
    json.dump(SEGMENT_PRICING_OVERRIDE, f, indent=2)

pol_df = pd.DataFrame(PRICING_POLICY).T.reset_index().rename(columns={"index": "Grade"})
print("\n  Pricing Policy Matrix:")
print(pol_df[["Grade","label","rate_floor","rate_ceil","tenure_min",
               "tenure_max","max_lti","lgd","risk_premium_base","approval"]].to_string(index=False))
pol_df.to_csv(os.path.join(POL_DIR, "pricing_policy_matrix.csv"), index=False)

print("\n  Segment Pricing Overrides:")
for seg, ovr in SEGMENT_PRICING_OVERRIDE.items():
    print(f"  {seg:<40} → rate_adj={ovr['rate_adj']:+.1f}% | "
          f"tenure_adj={ovr['tenure_adj']:+d}m | limit_adj=×{ovr['limit_adj']}")

log.info("Pricing policy matrix and segment overrides saved.")


══════════════════════════════════════════════════════════════════════
  STEP 2 — BUSINESS RULE FRAMEWORK & PRICING POLICY MATRIX
══════════════════════════════════════════════════════════════════════

  Pricing Policy Matrix:
Grade          label rate_floor rate_ceil tenure_min tenure_max max_lti   lgd risk_premium_base      approval
    A          Prime       10.0      14.0          6         60     5.0   0.5               0.5  Auto-Approve
    B     Near-Prime       14.0      19.0          6         48     4.0  0.55               3.5  Auto-Approve
    C      Sub-Prime       19.0      26.0          3         36     3.0   0.6               8.0   Conditional
    D      High Risk       26.0      32.0          3         24     2.0  0.65              14.0 Manual Review
    E Very High Risk       32.0      36.0          3         12     1.0   0.7              18.0       Decline

  Segment Pricing Overrides:
  Prime Stable Borrowers                   → rate_adj=-1.0% | tenure_adj=+12m | li

In [16]:
# =============================================================================
# CELL 6 — STEP 3 & 4: Risk-Based Pricing Engine & Rate Optimization
# =============================================================================

section("STEP 3 & 4 — RISK-BASED PRICING ENGINE & INTEREST RATE OPTIMIZATION")

def compute_risk_premium(pd_val: float, lgd: float, tenure_months: int) -> float:
    """
    Annualised risk premium using survival-model cumulative PD.
    EL_rate(annual) = [1 - (1-PD)^years] × LGD / years × 100
    This correctly accounts for multi-year exposure duration.
    """
    pd_val  = np.clip(pd_val, 0.001, 0.999)
    years   = max(tenure_months / 12, 0.25)
    cum_pd  = 1 - (1 - pd_val) ** years
    return round(cum_pd * lgd / years * 100, 4)


def compute_behavioral_modifier(
    spending_volatility: float = 0.30,
    payment_regularity:  float = 0.80,
    app_engagement:      float = 10.0,
    financial_stress:    float = 0.30,
) -> float:
    """
    Behavioral pricing modifier: -2.0% to +3.0%.
    Stable repayment + high engagement → rate discount.
    Volatile spending + financial stress → rate premium.
    """
    score  = 0.0
    score += (spending_volatility - 0.30) * 4.0   # Volatility penalty
    score -= (payment_regularity  - 0.80) * 3.0   # Regularity discount
    score -= (min(app_engagement, 20) / 20) * 1.0  # Engagement discount
    score += (financial_stress    - 0.30) * 3.0   # Stress penalty
    return round(float(np.clip(score, -2.0, 3.0)), 2)


def price_loan(
    pd_score:            float,
    risk_grade:          str,
    loan_amount:         float,
    tenure_months:       int,
    annual_income:       float = None,
    spending_volatility: float = 0.30,
    payment_regularity:  float = 0.80,
    app_engagement:      float = 10.0,
    financial_stress:    float = 0.30,
    ml_persona:          str   = None,
    config:              dict  = PRICING_CONFIG,
    policy:              dict  = PRICING_POLICY,
    seg_override:        dict  = SEGMENT_PRICING_OVERRIDE,
) -> dict:
    """
    Full risk-based pricing engine.
    Inputs  : PD, grade, amount, tenure, income, behavioral signals, persona
    Outputs : rate, risk premium, EMI, affordability, net profit, RAROC
    """
    grade = str(risk_grade)[0].upper()
    grade = grade if grade in policy else "C"
    pol   = policy[grade]
    lgd   = pol["lgd"]

    # ── 1. Risk premium (PD-driven) ────────────────────────────────────────
    risk_premium = compute_risk_premium(pd_score, lgd, tenure_months)

    # ── 2. Base rate = CoF + ops + risk + margin ───────────────────────────
    raw_rate = (
        config["base_rate"]
        + config["operational_spread"]
        + risk_premium
        + config["target_margin"]
    )

    # ── 3. Behavioral modifier ─────────────────────────────────────────────
    beh_mod = compute_behavioral_modifier(
        spending_volatility, payment_regularity, app_engagement, financial_stress
    )
    # Cap behavioral modifier by grade policy
    beh_mod  = float(np.clip(beh_mod, -2.0, pol["behavioral_mod_cap"]))
    raw_rate += beh_mod

    # ── 4. Segment override ────────────────────────────────────────────────
    seg_adj = 0.0
    if ml_persona and ml_persona in seg_override:
        seg_adj   = seg_override[ml_persona].get("rate_adj", 0.0)
        raw_rate += seg_adj

    # ── 5. Clamp to policy bounds ──────────────────────────────────────────
    recommended_rate = round(float(np.clip(raw_rate, pol["rate_floor"], pol["rate_ceil"])), 2)

    # ── 6. EMI calculation ─────────────────────────────────────────────────
    monthly_rate = recommended_rate / 100 / 12
    n = int(tenure_months)
    if monthly_rate > 0:
        emi = loan_amount * monthly_rate * (1 + monthly_rate)**n / ((1 + monthly_rate)**n - 1)
    else:
        emi = loan_amount / max(n, 1)

    monthly_income = annual_income / 12 if annual_income and annual_income > 0 else None
    emi_ratio      = emi / monthly_income if monthly_income else None
    affordability_ok = (emi_ratio < config["emi_burden_ceiling"]) if emi_ratio else True

    # ── 7. Profitability snapshot ──────────────────────────────────────────
    total_interest = emi * n - loan_amount
    years          = n / 12
    cum_pd         = 1 - (1 - pd_score) ** years
    expected_loss  = cum_pd * lgd * loan_amount
    net_profit     = total_interest - expected_loss
    # Simplified RAROC: net income / risk-weighted capital (EL × 12.5 × capital%)
    raroc = net_profit / max(expected_loss * 12.5 * 0.08, 1)

    return {
        "risk_grade":         grade,
        "pd_score":           round(pd_score, 4),
        "recommended_rate":   recommended_rate,
        "risk_premium_pct":   round(risk_premium, 3),
        "behavioral_mod":     round(beh_mod, 2),
        "segment_adj":        round(seg_adj, 2),
        "lgd_used":           lgd,
        "emi":                round(emi, 2),
        "emi_to_income":      round(emi_ratio, 4) if emi_ratio else None,
        "affordability_ok":   bool(affordability_ok),
        "total_interest":     round(total_interest, 2),
        "expected_loss":      round(expected_loss, 2),
        "net_profit":         round(net_profit, 2),
        "raroc":              round(raroc, 4),
        "approval":           pol["approval"],
    }


# ── Demo: 4 sample borrowers ───────────────────────────────────────────────
demo_cases = [
    {"label": "Prime borrower",      "pd": 0.03, "grade": "A", "amount": 300000, "tenure": 36, "income": 800000},
    {"label": "Near-prime borrower", "pd": 0.08, "grade": "B", "amount": 200000, "tenure": 24, "income": 500000},
    {"label": "Sub-prime borrower",  "pd": 0.15, "grade": "C", "amount": 100000, "tenure": 18, "income": 300000},
    {"label": "High-risk borrower",  "pd": 0.28, "grade": "D", "amount":  50000, "tenure": 12, "income": 200000},
]
print("\n  Pricing Engine Demo:")
print(f"  {'Borrower Type':<25} | {'Rate':>6} | {'Risk Prem':>10} | {'EMI':>10} | {'Net Profit':>12} | {'RAROC':>6}")
print("  " + "-"*80)
for c in demo_cases:
    r = price_loan(c["pd"], c["grade"], c["amount"], c["tenure"], c["income"])
    print(f"  {c['label']:<25} | {r['recommended_rate']:>5.1f}% | "
          f"{r['risk_premium_pct']:>9.2f}% | "
          f"₹{r['emi']:>8,.0f} | "
          f"₹{r['net_profit']:>10,.0f} | "
          f"{r['raroc']:>5.2f}")

log.info("Risk-based pricing engine defined and validated.")


══════════════════════════════════════════════════════════════════════
  STEP 3 & 4 — RISK-BASED PRICING ENGINE & INTEREST RATE OPTIMIZATION
══════════════════════════════════════════════════════════════════════

  Pricing Engine Demo:
  Borrower Type             |   Rate |  Risk Prem |        EMI |   Net Profit |  RAROC
  --------------------------------------------------------------------------------
  Prime borrower            |  13.5% |      1.46% | ₹  10,175 | ₹    53,193 |  4.06
  Near-prime borrower       |  16.2% |      4.22% | ₹   9,814 | ₹    18,632 |  1.10
  Sub-prime borrower        |  20.6% |      8.65% | ₹   6,508 | ₹     4,157 |  0.32
  High-risk borrower        |  30.2% |     18.20% | ₹   4,879 | ₹      -549 | -0.06
14:26:36 | INFO     | Risk-based pricing engine defined and validated.


In [17]:
# =============================================================================
# CELL 7 — Apply Pricing Engine to Full Portfolio
# =============================================================================

section("APPLYING PRICING ENGINE TO FULL PORTFOLIO")

# Extract arrays
pd_scores    = get_col(df, "pd_score",                     0.15)
grades       = df["risk_grade_clean"].fillna("C").values
loan_amounts = get_col(df, "loan_amount",                  100000)
tenures      = get_col(df, "loan_tenure_months",           24).astype(int)
incomes      = get_col(df, "annual_income",                400000)
sv           = get_col(df, "spending_volatility_index",    0.30)
pr           = get_col(df, "payment_regularization_score", 0.80)
ae           = get_col(df, "app_usage_mean",               10.0)
fs           = get_col(df, "financial_stress_index",       0.30)
personas     = df["ml_persona"].values if "ml_persona" in df.columns else np.full(len(df), None)

log.info("[Portfolio] Pricing %d loans …", len(df))
t0 = time.time()
pricing_results = []
for i in range(len(df)):
    r = price_loan(
        pd_score            = float(pd_scores[i]),
        risk_grade          = str(grades[i]),
        loan_amount         = float(loan_amounts[i]),
        tenure_months       = int(tenures[i]),
        annual_income       = float(incomes[i]),
        spending_volatility = float(sv[i]),
        payment_regularity  = float(pr[i]),
        app_engagement      = float(ae[i]),
        financial_stress    = float(fs[i]),
        ml_persona          = personas[i] if personas[i] else None,
    )
    pricing_results.append(r)

price_df = pd.DataFrame(pricing_results)
elapsed  = time.time() - t0

# Attach identifiers and original columns
for col in ["loan_id", "customer_id", "default_flag", "loan_type",
             "acquisition_channel", "ml_persona", "rule_segment"]:
    if col in df.columns:
        price_df[col] = df[col].values

price_df["loan_amount"]    = loan_amounts
price_df["tenure_months"]  = tenures
price_df["annual_income"]  = incomes

log.info("Portfolio pricing complete: %d loans in %.1fs.", len(price_df), elapsed)
print(f"\n  Priced {len(price_df):,} loans in {elapsed:.1f}s")
print(f"  Avg recommended rate : {price_df['recommended_rate'].mean():.2f}%")
print(f"  Avg risk premium     : {price_df['risk_premium_pct'].mean():.2f}%")
print(f"  Avg behavioral mod   : {price_df['behavioral_mod'].mean():+.3f}%")
print(f"  Avg net profit/loan  : ₹{price_df['net_profit'].mean():,.0f}")
print(f"  Avg RAROC            : {price_df['raroc'].mean():.3f}")
print(f"  Affordability OK     : {price_df['affordability_ok'].mean()*100:.1f}%")

price_df.to_csv(os.path.join(DASH_DIR, "portfolio_pricing_output.csv"), index=False)
log.info("Portfolio pricing output saved.")


══════════════════════════════════════════════════════════════════════
  APPLYING PRICING ENGINE TO FULL PORTFOLIO
══════════════════════════════════════════════════════════════════════
14:26:37 | INFO     | [Portfolio] Pricing 65000 loans …
14:26:40 | INFO     | Portfolio pricing complete: 65000 loans in 3.0s.

  Priced 65,000 loans in 3.0s
  Avg recommended rate : 34.21%
  Avg risk premium     : 29.83%
  Avg behavioral mod   : -0.497%
  Avg net profit/loan  : ₹33,911
  Avg RAROC            : -0.088
  Affordability OK     : 56.4%
14:26:41 | INFO     | Portfolio pricing output saved.


In [18]:
# =============================================================================
# CELL 8 — STEP 5 & 6: Tenure & Credit Limit Optimization
# =============================================================================

section("STEP 5 & 6 — TENURE & CREDIT LIMIT OPTIMIZATION")

def optimize_tenure(
    pd_score:      float,
    risk_grade:    str,
    loan_amount:   float,
    annual_income: float,
    current_rate:  float = None,
) -> dict:
    """
    Optimal tenure = tenure that maximises net profit subject to:
    1. EMI/Income ≤ 45%
    2. Within grade-specific policy tenure bounds
    3. Minimises cumulative expected loss
    Steps through every 3-month increment within the policy window.
    """
    grade = str(risk_grade)[0].upper()
    grade = grade if grade in PRICING_POLICY else "C"
    pol   = PRICING_POLICY[grade]
    lgd   = pol["lgd"]
    rate  = current_rate if current_rate else (pol["rate_floor"] + pol["rate_ceil"]) / 2
    mr    = rate / 100 / 12
    mi    = annual_income / 12 if annual_income and annual_income > 0 else loan_amount / 3

    best = {"tenure": pol["tenure_min"], "net_profit": -np.inf,
             "emi": None, "emi_ratio": None,
             "total_interest": None, "expected_loss": None}

    for t in range(pol["tenure_min"], pol["tenure_max"] + 1, 3):
        emi = (loan_amount * mr * (1+mr)**t / ((1+mr)**t - 1)) if mr > 0 else loan_amount / t
        er  = emi / mi if mi > 0 else 1.0
        if er > PRICING_CONFIG["emi_burden_ceiling"]:
            continue
        total_int = emi * t - loan_amount
        cum_pd    = 1 - (1 - pd_score) ** (t / 12)
        el        = cum_pd * lgd * loan_amount
        np_val    = total_int - el
        if np_val > best["net_profit"]:
            best = {
                "tenure":          t,
                "emi":             round(emi, 2),
                "emi_ratio":       round(er, 4),
                "total_interest":  round(total_int, 2),
                "expected_loss":   round(el, 2),
                "net_profit":      round(np_val, 2),
            }
    return best


def optimize_credit_limit(
    pd_score:      float,
    risk_grade:    str,
    annual_income: float,
    clv:           float = None,
    digital_trust: float = 0.5,
    repayment_consistency: float = 0.8,
    ml_persona:    str   = None,
) -> dict:
    """
    Dynamic credit limit = f(income, risk grade, CLV, behavioral trust).
    Base = monthly_income × grade_multiplier.
    Adjusted up for high CLV / high trust, down for risk.
    """
    grade = str(risk_grade)[0].upper()
    grade = grade if grade in PRICING_POLICY else "C"
    pol   = PRICING_POLICY[grade]
    mi    = annual_income / 12 if annual_income and annual_income > 0 else 30000

    base_limit = mi * pol["limit_multiplier"]
    # LTI cap
    base_limit = min(base_limit, annual_income * pol["max_lti"]) if annual_income else base_limit

    # CLV uplift
    clv_factor = 1.0
    if clv and clv > 500000: clv_factor = 1.20
    elif clv and clv > 250000: clv_factor = 1.10

    # Behavioral trust adjustment (±30% headroom)
    trust_adj = np.clip(
        1 + (digital_trust - 0.5) * 0.30 + (repayment_consistency - 0.8) * 0.20,
        0.70, 1.30
    )

    # Segment override
    seg_limit_adj = 1.0
    if ml_persona and ml_persona in SEGMENT_PRICING_OVERRIDE:
        seg_limit_adj = SEGMENT_PRICING_OVERRIDE[ml_persona].get("limit_adj", 1.0)

    recommended = round((base_limit * clv_factor * trust_adj * seg_limit_adj) / 10000) * 10000

    return {
        "recommended_limit": int(recommended),
        "base_limit":        int(base_limit),
        "clv_factor":        round(clv_factor, 2),
        "trust_adjustment":  round(float(trust_adj), 3),
        "seg_limit_adj":     round(seg_limit_adj, 2),
        "lti_ratio":         round(recommended / annual_income, 3) if annual_income else None,
    }


# ── Apply to full portfolio ────────────────────────────────────────────────
clv_vals = get_col(df, "customer_lifetime_value",      150000)
dt_vals  = get_col(df, "digital_trust_score",          0.50)
rc_vals  = get_col(df, "repayment_consistency_score",  0.80)

tenure_opts, limit_opts = [], []
for i in range(len(df)):
    tenure_opts.append(optimize_tenure(
        float(pd_scores[i]), str(grades[i]),
        float(loan_amounts[i]), float(incomes[i]),
        current_rate=float(price_df["recommended_rate"].iloc[i])
    ))
    limit_opts.append(optimize_credit_limit(
        float(pd_scores[i]), str(grades[i]),
        float(incomes[i]), float(clv_vals[i]),
        float(dt_vals[i]), float(rc_vals[i]),
        ml_persona=personas[i] if personas[i] else None
    ))

tenure_df = pd.DataFrame(tenure_opts)
limit_df  = pd.DataFrame(limit_opts)
price_df["optimal_tenure"]  = tenure_df["tenure"].values
price_df["optimal_emi"]     = tenure_df["emi"].values
price_df["optimal_limit"]   = limit_df["recommended_limit"].values
price_df["lti_ratio"]       = limit_df["lti_ratio"].values

print(f"\n  Tenure Optimization:")
print(f"  Avg optimal tenure           : {price_df['optimal_tenure'].mean():.1f} months")
print(f"  Avg current tenure in data   : {tenures.mean():.1f} months")
print(f"  Avg optimal EMI              : ₹{price_df['optimal_emi'].mean():,.0f}")
print(f"\n  Credit Limit Optimization:")
print(f"  Avg recommended limit        : ₹{price_df['optimal_limit'].mean():,.0f}")
print(f"  Avg requested loan amount    : ₹{loan_amounts.mean():,.0f}")

# Tenure optimization by grade
tenure_grade = price_df.groupby("risk_grade").agg(
    avg_optimal_tenure=("optimal_tenure", "mean"),
    avg_optimal_limit =("optimal_limit",  "mean"),
    avg_lti           =("lti_ratio",      "mean"),
).reset_index()
print("\n  Optimal Tenure & Limit by Grade:")
print(tenure_grade.to_string(index=False))
tenure_grade.to_csv(os.path.join(RPT_DIR, "tenure_limit_by_grade.csv"), index=False)
log.info("Tenure and credit limit optimization complete.")


══════════════════════════════════════════════════════════════════════
  STEP 5 & 6 — TENURE & CREDIT LIMIT OPTIMIZATION
══════════════════════════════════════════════════════════════════════

  Tenure Optimization:
  Avg optimal tenure           : 3.9 months
  Avg current tenure in data   : 30.4 months
  Avg optimal EMI              : ₹8,045

  Credit Limit Optimization:
  Avg recommended limit        : ₹20,148
  Avg requested loan amount    : ₹350,184

  Optimal Tenure & Limit by Grade:
risk_grade  avg_optimal_tenure  avg_optimal_limit  avg_lti
         B           40.500000       74821.428571 0.187054
         D            9.222222       35925.925926 0.089815
         E            3.805540       20000.000000 0.050000
14:26:47 | INFO     | Tenure and credit limit optimization complete.


In [19]:
# =============================================================================
# CELL 9 — STEP 7 & 8: Full Profitability Modeling & RAROC
# =============================================================================

section("STEP 7 & 8 — FULL PROFITABILITY MODELING & RAROC")

# Acquisition cost by channel (real fintech benchmarks)
CAC_BY_CHANNEL = {
    "Organic Search": 500,  "App Store": 600,  "Referral": 800,
    "Social Media": 1200,   "DSA Partner": 2000, "Bank Branch": 2500,
}
DEFAULT_CAC    = 1000
SERVICING_PA   = 500    # Annual servicing cost per loan (₹)
CAPITAL_RATIO  = 0.08   # 8% regulatory capital requirement

def compute_full_profitability(row: pd.Series) -> dict:
    """
    Full loan-level P&L:
    Net Income = Gross Interest - EL - Servicing - CAC - Capital Cost
    RAROC = Net Income / Risk-Weighted Capital
    NIM   = (Gross Interest - EL) / Loan Amount / Years × 100
    """
    amt    = float(row.get("loan_amount",        100000))
    rate   = float(row.get("recommended_rate",   18.0))
    n      = int(row.get("tenure_months",         24))
    pd_val = float(row.get("pd_score",            0.15))
    lgd    = float(row.get("lgd_used",            0.60))
    chan   = row.get("acquisition_channel", None)
    cac    = CAC_BY_CHANNEL.get(chan, DEFAULT_CAC) if chan else DEFAULT_CAC
    years  = n / 12

    mr  = rate / 100 / 12
    emi = amt * mr * (1+mr)**n / ((1+mr)**n - 1) if mr > 0 else amt / n
    gross_interest = emi * n - amt
    servicing_cost = SERVICING_PA * years
    cum_pd         = 1 - (1 - pd_val) ** years
    expected_loss  = cum_pd * lgd * amt
    capital_cost   = amt * CAPITAL_RATIO * years * (rate / 100)
    net_income     = gross_interest - expected_loss - servicing_cost - cac - capital_cost
    # Risk-Weighted Assets ≈ EL × 12.5; Capital = RWA × 8%
    rwa            = expected_loss * 12.5
    raroc          = net_income / max(rwa * 0.08, 1)
    nim            = (gross_interest - expected_loss) / max(amt, 1) / max(years, 0.25) * 100

    return {
        "gross_interest":  round(gross_interest, 2),
        "expected_loss":   round(expected_loss, 2),
        "servicing_cost":  round(servicing_cost, 2),
        "cac":             round(cac, 2),
        "capital_cost":    round(capital_cost, 2),
        "net_income":      round(net_income, 2),
        "raroc":           round(raroc, 4),
        "nim_pct":         round(nim, 3),
        "profitable":      int(net_income > 0),
    }


# Apply to portfolio
profit_results = price_df.apply(compute_full_profitability, axis=1)
profit_df = pd.DataFrame(list(profit_results))
for col in profit_df.columns:
    price_df[col] = profit_df[col].values

print("\n  Portfolio Profitability Summary (₹ Crore):")
print(f"  Gross interest income : ₹{price_df['gross_interest'].sum()/1e7:.2f} Cr")
print(f"  Expected loss         : ₹{price_df['expected_loss'].sum()/1e7:.2f} Cr")
print(f"  Servicing cost        : ₹{price_df['servicing_cost'].sum()/1e7:.2f} Cr")
print(f"  Acquisition cost      : ₹{price_df['cac'].sum()/1e7:.2f} Cr")
print(f"  Capital cost          : ₹{price_df['capital_cost'].sum()/1e7:.2f} Cr")
print(f"  Net income            : ₹{price_df['net_income'].sum()/1e7:.2f} Cr")
print(f"  Avg RAROC             : {price_df['raroc'].mean():.4f}")
print(f"  Avg NIM               : {price_df['nim_pct'].mean():.2f}%")
print(f"  % Profitable loans    : {price_df['profitable'].mean()*100:.1f}%")

# Grade-level profitability
grade_profit = price_df.groupby("risk_grade").agg(
    count          = ("net_income",      "count"),
    avg_rate       = ("recommended_rate", "mean"),
    avg_net_income = ("net_income",       "mean"),
    avg_raroc      = ("raroc",            "mean"),
    avg_nim        = ("nim_pct",          "mean"),
    pct_profitable = ("profitable",       "mean"),
    total_el       = ("expected_loss",    "sum"),
    total_ni       = ("net_income",       "sum"),
).reset_index()
grade_profit["pct_profitable"] = (grade_profit["pct_profitable"] * 100).round(1)
grade_profit["total_ni_cr"]    = (grade_profit["total_ni"] / 1e7).round(2)
grade_profit["total_el_cr"]    = (grade_profit["total_el"] / 1e7).round(2)
print("\n  Profitability by Risk Grade:")
print(grade_profit[["risk_grade","count","avg_rate","avg_net_income",
                     "avg_raroc","avg_nim","pct_profitable",
                     "total_ni_cr","total_el_cr"]].to_string(index=False))
grade_profit.to_csv(os.path.join(RPT_DIR, "profitability_by_grade.csv"), index=False)
price_df.to_csv(os.path.join(DASH_DIR, "portfolio_pricing_full.csv"), index=False)
log.info("Profitability modeling complete.")


══════════════════════════════════════════════════════════════════════
  STEP 7 & 8 — FULL PROFITABILITY MODELING & RAROC
══════════════════════════════════════════════════════════════════════

  Portfolio Profitability Summary (₹ Crore):
  Gross interest income : ₹1596.36 Cr
  Expected loss         : ₹1375.94 Cr
  Servicing cost        : ₹8.24 Cr
  Acquisition cost      : ₹6.50 Cr
  Capital cost          : ₹205.72 Cr
  Net income            : ₹-0.04 Cr
  Avg RAROC             : -0.4741
  Avg NIM               : -7.55%
  % Profitable loans    : 23.9%

  Profitability by Risk Grade:
risk_grade  count  avg_rate  avg_net_income  avg_raroc   avg_nim  pct_profitable  total_ni_cr  total_el_cr
         B    168 14.835060    48481.678214  -1.572712  5.511012            80.4         0.81         0.52
         D     27 27.605556    42296.762222  -0.581026  1.903222            51.9         0.11         0.51
         E  64805 34.263530     -149.288386  -0.471168 -7.590689            23.7        -

In [20]:
# =============================================================================
# CELL 10 — STEP 9 & 10: Segment-Based & Behavioral Pricing Intelligence
# =============================================================================

section("STEP 9 & 10 — SEGMENT-BASED & BEHAVIORAL PRICING INTELLIGENCE")

# ── Segment-level pricing analytics ───────────────────────────────────────
seg_col = None
for c in ["ml_persona", "rule_segment", "risk_grade"]:
    if c in price_df.columns and price_df[c].nunique() > 1:
        seg_col = c; break

seg_pricing = None
if seg_col:
    seg_pricing = price_df.groupby(seg_col).agg(
        count            = ("recommended_rate", "count"),
        avg_rate         = ("recommended_rate",  "mean"),
        avg_risk_premium = ("risk_premium_pct",  "mean"),
        avg_beh_mod      = ("behavioral_mod",    "mean"),
        avg_net_income   = ("net_income",        "mean"),
        avg_raroc        = ("raroc",             "mean"),
        avg_nim          = ("nim_pct",           "mean"),
        pct_profitable   = ("profitable",        "mean"),
        avg_pd           = ("pd_score",          "mean"),
    ).reset_index()
    seg_pricing["pct_profitable"] = (seg_pricing["pct_profitable"] * 100).round(1)
    seg_pricing.to_csv(os.path.join(RPT_DIR, f"segment_pricing_{seg_col}.csv"), index=False)
    print(f"\n  Segment Pricing Analytics (grouped by: {seg_col}):")
    print(seg_pricing[[seg_col, "count", "avg_rate", "avg_risk_premium",
                        "avg_beh_mod", "avg_net_income", "avg_raroc",
                        "pct_profitable"]].to_string(index=False))

# ── Behavioral modifier deep-dive ─────────────────────────────────────────
beh_vals = price_df["behavioral_mod"]
print(f"\n  Behavioral Modifier Distribution:")
print(f"  Mean    : {beh_vals.mean():+.3f}%")
print(f"  Min     : {beh_vals.min():+.3f}%  ← max discount (most stable borrower)")
print(f"  Max     : {beh_vals.max():+.3f}%  ← max premium (most volatile borrower)")
print(f"  % with discount (beh_mod < 0) : {(beh_vals < 0).mean()*100:.1f}%")
print(f"  % with premium  (beh_mod > 0) : {(beh_vals > 0).mean()*100:.1f}%")
print(f"  % neutral                      : {(beh_vals == 0).mean()*100:.1f}%")

# ── Channel-level pricing & profitability ─────────────────────────────────
if "acquisition_channel" in price_df.columns:
    chan_pricing = price_df.groupby("acquisition_channel").agg(
        count         = ("recommended_rate", "count"),
        avg_rate      = ("recommended_rate",  "mean"),
        avg_net_income= ("net_income",        "mean"),
        avg_cac       = ("cac",              "mean"),
        avg_raroc     = ("raroc",             "mean"),
        pct_profitable= ("profitable",        "mean"),
    ).reset_index()
    chan_pricing["pct_profitable"] = (chan_pricing["pct_profitable"]*100).round(1)
    chan_pricing.to_csv(os.path.join(RPT_DIR, "channel_pricing_profitability.csv"), index=False)
    print("\n  Channel Pricing & Profitability:")
    print(chan_pricing.sort_values("avg_raroc", ascending=False).to_string(index=False))

log.info("Segment and behavioral pricing analytics complete.")


══════════════════════════════════════════════════════════════════════
  STEP 9 & 10 — SEGMENT-BASED & BEHAVIORAL PRICING INTELLIGENCE
══════════════════════════════════════════════════════════════════════

  Segment Pricing Analytics (grouped by: risk_grade):
risk_grade  count  avg_rate  avg_risk_premium  avg_beh_mod  avg_net_income  avg_raroc  pct_profitable
         B    168 14.835060          2.807875    -0.625536    48481.678214  -1.572712            80.4
         D     27 27.605556         14.594815    -1.752593    42296.762222  -0.581026            51.9
         E  64805 34.263530         29.906019    -0.495808     -149.288386  -0.471168            23.7

  Behavioral Modifier Distribution:
  Mean    : -0.497%
  Min     : -2.000%  ← max discount (most stable borrower)
  Max     : +3.000%  ← max premium (most volatile borrower)
  % with discount (beh_mod < 0) : 68.8%
  % with premium  (beh_mod > 0) : 30.9%
  % neutral                      : 0.3%
14:26:51 | INFO     | Segment and 

In [21]:
# =============================================================================
# CELL 11 — STEP 11: Approval-Pricing Integration
# =============================================================================

section("STEP 11 — APPROVAL-PRICING INTEGRATION")

def integrated_underwriting_decision(
    pd_score:         float,
    risk_grade:       str,
    net_income:       float,
    raroc:            float,
    affordability_ok: bool,
    beh_mod:          float = 0.0,
    optimal_limit:    int   = 100000,
    requested_amount: float = None,
) -> dict:
    """
    Integrated approval + pricing decision with 5-tier output.
    Combines: risk grade, PD, profitability, affordability, behavioral signal.
    """
    grade = str(risk_grade)[0].upper()

    # ── Tier 1: Hard declines ─────────────────────────────────────────────
    if grade == "E" or pd_score > 0.55:
        return {"decision": "DECLINE",
                "reason":   f"Risk grade E or PD {pd_score:.1%} > 55% policy ceiling.",
                "recommended_limit": 0, "pricing_action": "N/A"}

    if not affordability_ok:
        return {"decision": "DECLINE",
                "reason":   "EMI/income ratio exceeds 45% affordability ceiling.",
                "recommended_limit": 0, "pricing_action": "N/A"}

    # ── Tier 2: Prime auto-approve ────────────────────────────────────────
    if grade in ["A", "B"] and net_income > 0 and raroc > 0.5:
        rec_limit = min(optimal_limit, int(requested_amount)) if requested_amount else optimal_limit
        action    = "APPROVE — Standard Rate" if grade == "A" else "APPROVE — Standard Rate"
        if requested_amount and requested_amount > optimal_limit:
            action = "APPROVE — Reduced Exposure (amount capped at limit)"
        return {"decision": "APPROVE", "pricing_action": action,
                "recommended_limit": int(rec_limit),
                "reason": f"Grade {grade} | PD={pd_score:.3f} | RAROC={raroc:.2f}"}

    # ── Tier 3: Sub-prime conditional ────────────────────────────────────
    if grade == "C" and net_income > 0:
        return {"decision": "CONDITIONAL",
                "pricing_action": "APPROVE — Premium Pricing + Shorter Tenure",
                "recommended_limit": int(optimal_limit * 0.75),
                "reason": f"Grade C | BehMod={beh_mod:+.1f}% | Net={net_income:,.0f}"}

    # ── Tier 4: High-risk manual review ──────────────────────────────────
    if grade == "D" or raroc < 0:
        return {"decision": "MANUAL_REVIEW",
                "pricing_action": "REFER — Collateral / co-borrower required",
                "recommended_limit": int(optimal_limit * 0.50),
                "reason": f"Grade D or RAROC={raroc:.2f} < 0"}

    # ── Tier 5: Edge cases ────────────────────────────────────────────────
    return {"decision": "MANUAL_REVIEW",
             "pricing_action": "MANUAL — Edge case",
             "recommended_limit": int(optimal_limit * 0.60),
             "reason": "Fallback — manual underwriter review required."}


# Apply to portfolio
decisions = []
for i, row in price_df.iterrows():
    d = integrated_underwriting_decision(
        pd_score         = row["pd_score"],
        risk_grade       = row["risk_grade"],
        net_income       = row["net_income"],
        raroc            = row["raroc"],
        affordability_ok = row.get("affordability_ok", True),
        beh_mod          = row.get("behavioral_mod", 0.0),
        optimal_limit    = row.get("optimal_limit", 100000),
        requested_amount = row.get("loan_amount"),
    )
    decisions.append(d)

dec_df = pd.DataFrame(decisions)
price_df["integrated_decision"] = dec_df["decision"].values
price_df["integrated_action"]   = dec_df["pricing_action"].values
price_df["integrated_limit"]    = dec_df["recommended_limit"].values

print("\n  Integrated Decision Distribution:")
print(price_df["integrated_decision"].value_counts().to_string())
print(f"\n  Approval Rate (APPROVE + CONDITIONAL): "
      f"{price_df['integrated_decision'].isin(['APPROVE','CONDITIONAL']).mean()*100:.1f}%")
print(f"  Pure Approval Rate:   {(price_df['integrated_decision']=='APPROVE').mean()*100:.1f}%")
print(f"  Decline Rate:         {(price_df['integrated_decision']=='DECLINE').mean()*100:.1f}%")
print(f"  Manual Review Rate:   {(price_df['integrated_decision']=='MANUAL_REVIEW').mean()*100:.1f}%")

# Decision × Grade cross-tab
dec_grade = pd.crosstab(price_df["risk_grade"], price_df["integrated_decision"],
                          normalize="index").round(3) * 100
print("\n  Decision Distribution by Grade (%):")
print(dec_grade.to_string())
dec_df.to_csv(os.path.join(RPT_DIR, "integrated_underwriting_decisions.csv"), index=False)
price_df.to_csv(os.path.join(DASH_DIR, "portfolio_pricing_full.csv"), index=False)
log.info("Approval-pricing integration complete. Part 1 complete.")

print("\n" + "═"*70)
print("  MODULE 6 PART 1 — COMPLETE")
print("═"*70)
print(f"  Loans priced         : {len(price_df):,}")
print(f"  Avg rate             : {price_df['recommended_rate'].mean():.2f}%")
print(f"  Avg RAROC            : {price_df['raroc'].mean():.4f}")
print(f"  % Profitable         : {price_df['profitable'].mean()*100:.1f}%")
print(f"  Total net income     : ₹{price_df['net_income'].sum()/1e7:.2f} Cr")
print(f"  Output saved to      : {DASH_DIR}")
print("  → Load Part 2 to continue with scenarios, stress testing,")
print("    explainability, fairness, visual dashboards & production pipeline.")
print("═"*70)


══════════════════════════════════════════════════════════════════════
  STEP 11 — APPROVAL-PRICING INTEGRATION
══════════════════════════════════════════════════════════════════════

  Integrated Decision Distribution:
integrated_decision
DECLINE          64876
APPROVE             80
MANUAL_REVIEW       44

  Approval Rate (APPROVE + CONDITIONAL): 0.1%
  Pure Approval Rate:   0.1%
  Decline Rate:         99.8%
  Manual Review Rate:   0.1%

  Decision Distribution by Grade (%):
integrated_decision  APPROVE  DECLINE  MANUAL_REVIEW
risk_grade                                          
B                       47.6     33.3           19.0
D                        0.0     55.6           44.4
E                        0.0    100.0            0.0
14:27:00 | INFO     | Approval-pricing integration complete. Part 1 complete.

══════════════════════════════════════════════════════════════════════
  MODULE 6 PART 1 — COMPLETE
══════════════════════════════════════════════════════════════════════
 

In [22]:
# =============================================================================
# CELL 12 — STEP 13: Scenario Simulation Engine
# =============================================================================

section("STEP 13 — SCENARIO SIMULATION ENGINE")

SCENARIOS = {
    "Baseline": {
        "pd_shock":     0.00, "rate_adj":    0.00,
        "income_shock": 0.00, "vol_shock":   0.00,
        "description":  "Current conditions — no macro changes."
    },
    "Aggressive Growth": {
        "pd_shock":     +0.02, "rate_adj":   -2.00,
        "income_shock":  0.00, "vol_shock":  +0.05,
        "description":  "Rate cuts to drive approval volume; accepts higher PD."
    },
    "Conservative Lending": {
        "pd_shock":     -0.01, "rate_adj":   +2.00,
        "income_shock":  0.00, "vol_shock":  -0.05,
        "description":  "Tighter underwriting; fewer, higher-quality approvals."
    },
    "Economic Downturn": {
        "pd_shock":     +0.06, "rate_adj":   +1.50,
        "income_shock": -0.10, "vol_shock":  +0.15,
        "description":  "GDP contraction, income drop, rising delinquency."
    },
    "Unemployment Surge": {
        "pd_shock":     +0.10, "rate_adj":   +2.00,
        "income_shock": -0.20, "vol_shock":  +0.20,
        "description":  "+5pp unemployment; severe income and PD shock."
    },
    "Rate Cap Reduction": {
        "pd_shock":      0.00, "rate_adj":   -5.00,
        "income_shock":  0.00, "vol_shock":   0.00,
        "description":  "Regulator cuts rate ceiling by 5pp."
    },
    "Spending Shock": {
        "pd_shock":     +0.04, "rate_adj":   +1.00,
        "income_shock": -0.05, "vol_shock":  +0.25,
        "description":  "Inflation spike; household spending shock."
    },
}

sim_results = []

for scenario_name, shocks in SCENARIOS.items():
    stressed = []
    for i in range(len(df)):
        pd_s  = float(np.clip(float(pd_scores[i]) + shocks["pd_shock"], 0.001, 0.999))
        sv_s  = float(np.clip(float(sv[i]) + shocks["vol_shock"],  0.0, 1.0))
        inc_s = float(incomes[i]) * (1 + shocks["income_shock"])

        r = price_loan(
            pd_score            = pd_s,
            risk_grade          = str(grades[i]),
            loan_amount         = float(loan_amounts[i]),
            tenure_months       = int(tenures[i]),
            annual_income       = inc_s,
            spending_volatility = sv_s,
            payment_regularity  = float(pr[i]),
            app_engagement      = float(ae[i]),
            financial_stress    = float(fs[i]),
            ml_persona          = personas[i] if personas[i] else None,
        )
        # Apply rate shift (clamped to global policy bounds)
        adj_rate = float(np.clip(
            r["recommended_rate"] + shocks["rate_adj"],
            PRICING_CONFIG["min_rate"], PRICING_CONFIG["max_rate"]
        ))
        # Recompute profitability at adjusted rate
        n     = int(tenures[i])
        mr    = adj_rate / 100 / 12
        emi   = float(loan_amounts[i]) * mr * (1+mr)**n / ((1+mr)**n - 1) if mr > 0 else float(loan_amounts[i]) / n
        gross = emi * n - float(loan_amounts[i])
        lgd_s = PRICING_POLICY.get(str(grades[i])[0].upper(), PRICING_POLICY["C"])["lgd"]
        yrs   = n / 12
        el    = (1 - (1 - pd_s) ** yrs) * lgd_s * float(loan_amounts[i])
        ni    = gross - el - SERVICING_PA * yrs - DEFAULT_CAC

        stressed.append({"rate": adj_rate, "net_income": ni, "el": el,
                          "raroc": ni / max(el * 12.5 * 0.08, 1)})

    sr = pd.DataFrame(stressed)
    sim_results.append({
        "scenario":             scenario_name,
        "avg_rate":             round(sr["rate"].mean(), 2),
        "total_net_income_cr":  round(sr["net_income"].sum() / 1e7, 2),
        "total_el_cr":          round(sr["el"].sum() / 1e7, 2),
        "avg_raroc":            round(sr["raroc"].mean(), 4),
        "pct_profitable":       round((sr["net_income"] > 0).mean() * 100, 1),
        "description":          shocks["description"],
    })
    log.info("  Scenario '%s': rate=%.2f%% | EL=₹%.2fCr | NI=₹%.2fCr",
             scenario_name, sr["rate"].mean(),
             sr["el"].sum()/1e7, sr["net_income"].sum()/1e7)

sim_df = pd.DataFrame(sim_results)
baseline_ni = sim_df.loc[sim_df["scenario"]=="Baseline", "total_net_income_cr"].values[0]
baseline_el = sim_df.loc[sim_df["scenario"]=="Baseline", "total_el_cr"].values[0]
sim_df["ni_delta_cr"] = (sim_df["total_net_income_cr"] - baseline_ni).round(2)
sim_df["el_delta_cr"] = (sim_df["total_el_cr"]         - baseline_el).round(2)
sim_df["ni_delta_pct"]= ((sim_df["total_net_income_cr"] / max(baseline_ni, 0.001) - 1) * 100).round(1)

print("\n  Scenario Simulation Results:")
print(sim_df[["scenario","avg_rate","total_net_income_cr","total_el_cr",
               "avg_raroc","pct_profitable","ni_delta_cr","ni_delta_pct"]].to_string(index=False))
sim_df.to_csv(os.path.join(SIM_DIR, "scenario_simulation_results.csv"), index=False)
log.info("Scenario simulation complete.")


══════════════════════════════════════════════════════════════════════
  STEP 13 — SCENARIO SIMULATION ENGINE
══════════════════════════════════════════════════════════════════════
14:27:12 | INFO     |   Scenario 'Baseline': rate=34.21% | EL=₹1375.94Cr | NI=₹205.68Cr
14:27:16 | INFO     |   Scenario 'Aggressive Growth': rate=32.27% | EL=₹1392.61Cr | NI=₹79.47Cr
14:27:21 | INFO     |   Scenario 'Conservative Lending': rate=35.12% | EL=₹1367.19Cr | NI=₹309.45Cr
14:27:25 | INFO     |   Scenario 'Economic Downturn': rate=35.03% | EL=₹1422.87Cr | NI=₹236.79Cr
14:27:30 | INFO     |   Scenario 'Unemployment Surge': rate=35.26% | EL=₹1449.49Cr | NI=₹234.83Cr
14:27:34 | INFO     |   Scenario 'Rate Cap Reduction': rate=29.21% | EL=₹1375.94Cr | NI=₹-71.90Cr
14:27:38 | INFO     |   Scenario 'Spending Shock': rate=34.83% | EL=₹1408.22Cr | NI=₹229.36Cr

  Scenario Simulation Results:
            scenario  avg_rate  total_net_income_cr  total_el_cr  avg_raroc  pct_profitable  ni_delta_cr  ni_delta_

In [23]:
# =============================================================================
# CELL 13 — STEP 12: Portfolio-Level Pricing Optimization & Sensitivity
# =============================================================================

section("STEP 12 — PORTFOLIO-LEVEL PRICING OPTIMIZATION & SENSITIVITY")

def portfolio_pricing_sensitivity(
    price_df: pd.DataFrame,
    rate_shifts: list = [-3, -2, -1, 0, +1, +2, +3]
) -> pd.DataFrame:
    """
    Sweep ±N pp rate shifts across the full portfolio.
    Recomputes EMI, affordability, net income at each shift.
    """
    rows = []
    for shift in rate_shifts:
        adj_rates = np.clip(
            price_df["recommended_rate"] + shift,
            PRICING_CONFIG["min_rate"], PRICING_CONFIG["max_rate"]
        )
        mr   = adj_rates / 100 / 12
        n    = price_df["tenure_months"].fillna(24).astype(int)
        amt  = price_df["loan_amount"].fillna(100000)
        emis = amt * mr * (1+mr)**n / ((1+mr)**n - 1)
        gross= emis * n - amt
        el   = price_df["expected_loss"]
        ni   = gross - el - price_df["servicing_cost"] - price_df["cac"] - price_df["capital_cost"]
        mi   = price_df["annual_income"].fillna(400000) / 12
        aff  = (emis / mi) < PRICING_CONFIG["emi_burden_ceiling"]
        rows.append({
            "rate_shift_pp":     shift,
            "avg_rate":          round(adj_rates.mean(), 2),
            "total_ni_cr":       round(ni.sum() / 1e7, 2),
            "total_el_cr":       round(el.sum() / 1e7, 2),
            "pct_profitable":    round((ni > 0).mean() * 100, 1),
            "pct_affordable":    round(aff.mean() * 100, 1),
            "ni_vs_baseline_cr": 0.0,
        })
    df_out = pd.DataFrame(rows)
    base_ni = df_out.loc[df_out["rate_shift_pp"]==0, "total_ni_cr"].values[0]
    df_out["ni_vs_baseline_cr"] = (df_out["total_ni_cr"] - base_ni).round(2)
    return df_out


sens_df = portfolio_pricing_sensitivity(price_df)
print("\n  Portfolio Pricing Sensitivity (rate shift ±3pp):")
print(sens_df.to_string(index=False))
sens_df.to_csv(os.path.join(SIM_DIR, "pricing_sensitivity.csv"), index=False)

# ── Exposure concentration by grade ───────────────────────────────────────
grade_exposure = price_df.groupby("risk_grade").agg(
    count          = ("loan_amount", "count"),
    total_exposure = ("loan_amount", "sum"),
    total_ni       = ("net_income",  "sum"),
).reset_index()
grade_exposure["exposure_pct"] = (
    grade_exposure["total_exposure"] / grade_exposure["total_exposure"].sum() * 100
).round(2)
grade_exposure["ni_pct"] = (
    grade_exposure["total_ni"] / grade_exposure["total_ni"].sum() * 100
).round(2)
grade_exposure["concentration_flag"] = grade_exposure["exposure_pct"].apply(
    lambda x: "⚠ HIGH" if x > 40 else "✅ OK"
)
print("\n  Grade Exposure Concentration:")
print(grade_exposure[["risk_grade","count","exposure_pct","ni_pct","concentration_flag"]].to_string(index=False))
grade_exposure.to_csv(os.path.join(RPT_DIR, "grade_exposure_concentration.csv"), index=False)

# ── Optimal portfolio mix recommendation ──────────────────────────────────
print("\n  Portfolio Optimisation Recommendations:")
for _, row in grade_exposure.iterrows():
    if row["exposure_pct"] > 40:
        print(f"  ⚠  Grade {row['risk_grade']}: {row['exposure_pct']:.1f}% exposure — consider capping below 40%.")
    elif row["ni_pct"] > row["exposure_pct"]:
        print(f"  ✅  Grade {row['risk_grade']}: NI% ({row['ni_pct']:.1f}%) > Exposure% ({row['exposure_pct']:.1f}%) — scale up.")
    else:
        print(f"  ⚡  Grade {row['risk_grade']}: NI% ({row['ni_pct']:.1f}%) < Exposure% ({row['exposure_pct']:.1f}%) — review pricing.")

log.info("Portfolio sensitivity and exposure analysis complete.")


══════════════════════════════════════════════════════════════════════
  STEP 12 — PORTFOLIO-LEVEL PRICING OPTIMIZATION & SENSITIVITY
══════════════════════════════════════════════════════════════════════

  Portfolio Pricing Sensitivity (rate shift ±3pp):
 rate_shift_pp  avg_rate  total_ni_cr  total_el_cr  pct_profitable  pct_affordable  ni_vs_baseline_cr
            -3     31.21      -167.94      1375.94            15.2            58.1            -167.90
            -2     32.21      -112.41      1375.94            18.3            57.6            -112.37
            -1     33.21       -56.44      1375.94            21.2            57.0             -56.40
             0     34.21        -0.04      1375.94            23.9            56.4               0.00
             1     34.69        48.44      1375.94            26.5            56.0              48.48
             2     35.14        96.11      1375.94            29.0            55.6              96.15
             3     35.56    

In [24]:
# =============================================================================
# CELL 14 — STEP 14: Explainable Pricing Layer
# =============================================================================

section("STEP 14 — EXPLAINABLE PRICING LAYER")

def explain_pricing_decision(row: pd.Series) -> str:
    """
    Human-readable pricing explanation for CRO dashboards,
    audit trails, and customer-facing communications.
    """
    grade  = str(row.get("risk_grade",  "C"))
    rate   = float(row.get("recommended_rate", 18.0))
    rp     = float(row.get("risk_premium_pct",  5.0))
    bm     = float(row.get("behavioral_mod",    0.0))
    sa     = float(row.get("segment_adj",       0.0))
    pd_val = float(row.get("pd_score",         0.15))
    net_p  = float(row.get("net_income",        0.0))
    raroc  = float(row.get("raroc",             0.0))
    aff    = bool(row.get("affordability_ok",   True))
    persona= str(row.get("ml_persona", ""))

    base   = PRICING_CONFIG["base_rate"] + PRICING_CONFIG["operational_spread"] + PRICING_CONFIG["target_margin"]

    lines = [
        f"{'─'*60}",
        f"📋 PRICING DECISION EXPLANATION",
        f"   Risk Grade : {grade}  |  PD Score : {pd_val:.3f}  |  Final Rate : {rate:.2f}%",
        f"{'─'*60}",
        f"  RATE CONSTRUCTION:",
        f"    Base rate (CoF + Ops + Margin)  : {base:.2f}%",
        f"    + Risk premium (PD={pd_val:.3f})    : {rp:+.2f}%",
    ]

    if abs(bm) >= 0.01:
        bm_label = "behavioral discount" if bm < 0 else "behavioral premium"
        lines.append(f"    + {bm_label:<34}: {bm:+.2f}%")
    if abs(sa) >= 0.01:
        sa_label = "segment discount" if sa < 0 else "segment premium"
        seg_desc = f"({persona})" if persona else ""
        lines.append(f"    + {sa_label:<34}: {sa:+.2f}%  {seg_desc}")

    lines += [
        f"    = FINAL RATE                    : {rate:.2f}%",
        f"",
        f"  PROFITABILITY:",
        f"    Net income projection           : ₹{net_p:,.0f}",
        f"    RAROC                           : {raroc:.3f}",
        f"    Affordability                   : {'✅ Within 45% EMI ceiling' if aff else '❌ Exceeds ceiling'}",
        f"",
        f"  KEY DRIVERS:",
    ]

    drivers = []
    if pd_val > 0.25:   drivers.append(f"↑ High default probability ({pd_val:.1%}) drives risk premium")
    elif pd_val < 0.06: drivers.append(f"↓ Very low PD ({pd_val:.1%}) qualifies for prime rate")
    if rp > 10:         drivers.append(f"↑ Long-tenure cumulative EL inflates risk premium")
    if bm > 1.0:        drivers.append(f"↑ Elevated spending volatility / financial stress adds {bm:.1f}pp")
    if bm < -0.5:       drivers.append(f"↓ Consistent repayment and high engagement earns {abs(bm):.1f}pp discount")
    if sa < -0.5:       drivers.append(f"↓ Loyalty / persona segment discount of {abs(sa):.1f}pp applied")
    if sa > 1.0:        drivers.append(f"↑ High-risk segment premium of {sa:.1f}pp applied")
    if raroc < 0:       drivers.append(f"⚠ Negative RAROC — pricing may be inadequate for risk level")

    if not drivers:
        drivers.append("Standard pricing — no exceptional drivers.")

    for d in drivers:
        lines.append(f"    • {d}")

    return "\n".join(lines)


# ── Generate explanations for one sample per grade ────────────────────────
print("\n  Pricing Explanation Examples (one per grade):")
for g in sorted(price_df["risk_grade"].unique()):
    sub = price_df[price_df["risk_grade"] == g]
    if len(sub) > 0:
        print("\n" + explain_pricing_decision(sub.iloc[0]))

# ── Save explanations for 500-loan sample ─────────────────────────────────
price_df["pricing_explanation"] = price_df.apply(explain_pricing_decision, axis=1)

price_df[[
    "risk_grade", "pd_score", "recommended_rate",
    "risk_premium_pct", "behavioral_mod", "segment_adj",
    "net_income", "raroc", "affordability_ok", "pricing_explanation"
]].head(500).to_csv(os.path.join(EXP_DIR, "pricing_explanations_sample.csv"), index=False)

# ── SHAP-style contribution table for top 200 loans ───────────────────────
contrib_df = price_df[["risk_grade","pd_score","recommended_rate",
                         "risk_premium_pct","behavioral_mod","segment_adj"]].copy()
contrib_df["base_rate"]  = PRICING_CONFIG["base_rate"] + PRICING_CONFIG["operational_spread"] + PRICING_CONFIG["target_margin"]
contrib_df["rate_check"] = (contrib_df["base_rate"] + contrib_df["risk_premium_pct"]
                             + contrib_df["behavioral_mod"] + contrib_df["segment_adj"]).round(2)
contrib_df.head(200).to_csv(os.path.join(EXP_DIR, "pricing_contribution_table.csv"), index=False)

log.info("Explainable pricing layer complete.")
print("\n  ✅  Pricing explanations saved to explainability/")


══════════════════════════════════════════════════════════════════════
  STEP 14 — EXPLAINABLE PRICING LAYER
══════════════════════════════════════════════════════════════════════

  Pricing Explanation Examples (one per grade):

────────────────────────────────────────────────────────────
📋 PRICING DECISION EXPLANATION
   Risk Grade : B  |  PD Score : 0.053  |  Final Rate : 14.10%
────────────────────────────────────────────────────────────
  RATE CONSTRUCTION:
    Base rate (CoF + Ops + Margin)  : 12.50%
    + Risk premium (PD=0.053)    : +2.93%
    + behavioral discount               : -1.33%
    = FINAL RATE                    : 14.10%

  PROFITABILITY:
    Net income projection           : ₹6,736
    RAROC                           : 1.034
    Affordability                   : ❌ Exceeds ceiling

  KEY DRIVERS:
    • ↓ Very low PD (5.3%) qualifies for prime rate
    • ↓ Consistent repayment and high engagement earns 1.3pp discount

─────────────────────────────────────────────────

In [25]:
# =============================================================================
# CELL 15 — STEP 15: Fairness & Bias Analysis
# =============================================================================

section("STEP 15 — FAIRNESS & BIAS ANALYSIS")

def compute_fairness_metrics(
    df_in:    pd.DataFrame,
    group_col:str,
    rate_col: str = "recommended_rate",
    pd_col:   str = "pd_score",
) -> pd.DataFrame:
    """
    Pricing fairness test:
    Rates should be proportional to risk (PD).
    fairness_ratio = group (rate/PD) / portfolio (rate/PD)
    Ratio > 1.15 → over-priced vs risk; < 0.85 → under-priced vs risk.
    """
    if group_col not in df_in.columns:
        return pd.DataFrame()
    g = df_in.groupby(group_col).agg(
        count    = (rate_col, "count"),
        avg_rate = (rate_col, "mean"),
        avg_pd   = (pd_col,   "mean"),
        std_rate = (rate_col, "std"),
    ).reset_index()
    overall_rpd        = g["avg_rate"].mean() / max(g["avg_pd"].mean() * 100, 0.001)
    g["rate_per_pd"]   = (g["avg_rate"] / (g["avg_pd"] * 100 + 1e-6)).round(3)
    g["fairness_ratio"]= (g["rate_per_pd"] / max(overall_rpd, 1e-6)).round(3)
    g["bias_flag"]     = g["fairness_ratio"].apply(
        lambda x: "⚠ Overpriced" if x > 1.15 else "⚠ Underpriced" if x < 0.85 else "✅ Fair"
    )
    return g


FAIRNESS_GROUPS = {
    "risk_grade":          "Risk Grade",
    "ml_persona":          "ML Persona",
    "rule_segment":        "Rule Segment",
    "acquisition_channel": "Acquisition Channel",
    "loan_type":           "Loan Type",
}

fairness_summary = {}
for col, label in FAIRNESS_GROUPS.items():
    result = compute_fairness_metrics(price_df, col)
    if len(result):
        fairness_summary[col] = result
        result.to_csv(os.path.join(RPT_DIR, f"fairness_{col}.csv"), index=False)
        print(f"\n  Fairness — {label}:")
        print(result[[col,"count","avg_rate","avg_pd",
                        "rate_per_pd","fairness_ratio","bias_flag"]].to_string(index=False))

# ── Statistical test: ANOVA — are rates differentiated by grade? ──────────
if "risk_grade" in price_df.columns:
    anova_groups = [
        price_df[price_df["risk_grade"]==g]["recommended_rate"].dropna()
        for g in sorted(price_df["risk_grade"].unique())
        if len(price_df[price_df["risk_grade"]==g]) > 5
    ]
    if len(anova_groups) >= 2:
        f_stat, p_val = f_oneway(*anova_groups)
        sig = "✅ Rates ARE significantly risk-differentiated" if p_val < 0.05 \
              else "❌ Rates NOT risk-differentiated — pricing bias possible"
        print(f"\n  ANOVA (Rate vs Grade): F={f_stat:.2f} | p={p_val:.2e}")
        print(f"  {sig}")

# ── Fairness visualization ─────────────────────────────────────────────────
if "risk_grade" in fairness_summary:
    fr = fairness_summary["risk_grade"]
    fc = [PAL["danger"] if "Over" in b else PAL["warning"] if "Under" in b else PAL["success"]
           for b in fr["bias_flag"]]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Pricing Fairness Analysis", fontsize=13, fontweight="bold", color=PAL["primary"])

    axes[0].bar(fr["risk_grade"], fr["fairness_ratio"], color=fc)
    axes[0].axhline(1.0,  color="black", linestyle="-",  linewidth=1, label="Fair (ratio=1.0)")
    axes[0].axhline(1.15, color=PAL["danger"],  linestyle="--", linewidth=1, label="Overpriced limit")
    axes[0].axhline(0.85, color=PAL["warning"], linestyle="--", linewidth=1, label="Underpriced limit")
    axes[0].set_title("Fairness Ratio by Risk Grade\n(1.0 = perfectly proportional to PD)")
    axes[0].set_ylabel("Fairness Ratio")
    axes[0].legend(fontsize=8)

    axes[1].bar(fr["risk_grade"], fr["avg_rate"],
                color=[GRADE_COLORS.get(g, PAL["neutral"]) for g in fr["risk_grade"]])
    axes[1].set_title("Average Rate by Risk Grade")
    axes[1].set_ylabel("Avg Interest Rate (%)")
    for i, (v, dr) in enumerate(zip(fr["avg_rate"], fr["avg_pd"]*100)):
        axes[1].text(i, v+0.2, f"{v:.1f}%\nPD:{dr:.1f}%", ha="center", fontsize=8)

    plt.tight_layout()
    savefig("00_pricing_fairness.png")

log.info("Fairness and bias analysis complete.")


══════════════════════════════════════════════════════════════════════
  STEP 15 — FAIRNESS & BIAS ANALYSIS
══════════════════════════════════════════════════════════════════════

  Fairness — Risk Grade:
risk_grade  count  avg_rate   avg_pd  rate_per_pd  fairness_ratio     bias_flag
         B    168 14.835060 0.053300        2.783           3.327  ⚠ Overpriced
         D     27 27.605556 0.285681        0.966           1.155  ⚠ Overpriced
         E  64805 34.263530 0.578026        0.593           0.709 ⚠ Underpriced

  Fairness — Loan Type:
          loan_type  count  avg_rate   avg_pd  rate_per_pd  fairness_ratio bias_flag
               BNPL  12926 35.949123 0.576030        0.624           1.046    ✅ Fair
      Personal Loan  35916 34.036573 0.576205        0.591           0.991    ✅ Fair
SME Working Capital  16158 33.206448 0.577726        0.575           0.964    ✅ Fair

  ANOVA (Rate vs Grade): F=8908.49 | p=0.00e+00
  ✅ Rates ARE significantly risk-differentiated
14:28:54 | I

In [26]:
# =============================================================================
# CELL 16 — STEP 18: Pricing Stress Testing
# =============================================================================

section("STEP 18 — PRICING ADEQUACY STRESS TESTING")

STRESS_SCENARIOS = {
    "Baseline":       {"pd_mult": 1.00, "income_mult": 1.00},
    "Mild Stress":    {"pd_mult": 1.20, "income_mult": 0.95},
    "Moderate Stress":{"pd_mult": 1.50, "income_mult": 0.85},
    "Severe Stress":  {"pd_mult": 2.00, "income_mult": 0.75},
    "Extreme Stress": {"pd_mult": 3.00, "income_mult": 0.60},
}

stress_rows = []
for name, params in STRESS_SCENARIOS.items():
    s_pd  = np.clip(price_df["pd_score"] * params["pd_mult"], 0.001, 0.999)
    lgd_v = price_df["lgd_used"].fillna(0.60)
    amt   = price_df["loan_amount"].fillna(100000)
    n_v   = price_df["tenure_months"].fillna(24)
    yrs_v = n_v / 12

    stressed_el = (1 - (1 - s_pd) ** yrs_v) * lgd_v * amt
    stressed_ni = price_df["gross_interest"] - stressed_el \
                  - price_df["servicing_cost"] - price_df["cac"]

    # Pricing adequacy: current rate covers CoF + ops + stressed EL rate?
    stressed_el_rate = stressed_el / amt.replace(0, np.nan) * 100 / yrs_v.replace(0, np.nan)
    min_adequate_rate = (
        PRICING_CONFIG["base_rate"]
        + PRICING_CONFIG["operational_spread"]
        + stressed_el_rate.fillna(0)
    )
    adequate = price_df["recommended_rate"] >= min_adequate_rate

    stress_rows.append({
        "scenario":             name,
        "pd_multiplier":        params["pd_mult"],
        "avg_stressed_pd_pct":  round(s_pd.mean() * 100, 2),
        "stressed_el_cr":       round(stressed_el.sum() / 1e7, 2),
        "stressed_ni_cr":       round(stressed_ni.sum() / 1e7, 2),
        "pct_pricing_adequate": round(adequate.mean() * 100, 1),
        "pct_loans_negative_ni":round((stressed_ni < 0).mean() * 100, 1),
    })

stress_df = pd.DataFrame(stress_rows)
print("\n  Pricing Adequacy Under Stress:")
print(stress_df.to_string(index=False))
stress_df.to_csv(os.path.join(SIM_DIR, "pricing_stress_test.csv"), index=False)

# ── Stress visualization ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Pricing Adequacy — Stress Test", fontsize=13, fontweight="bold", color=PAL["primary"])

adq_colors = [
    PAL["success"] if v >= 75 else PAL["warning"] if v >= 50 else PAL["danger"]
    for v in stress_df["pct_pricing_adequate"]
]
ni_colors  = [
    PAL["success"] if v >= 0 else PAL["danger"]
    for v in stress_df["stressed_ni_cr"]
]

axes[0].bar(stress_df["scenario"], stress_df["pct_pricing_adequate"], color=adq_colors)
axes[0].axhline(75, color=PAL["warning"], linestyle="--", linewidth=1.5, label="75% adequacy threshold")
axes[0].set_title("% Loans: Rate Covers Stressed Expected Loss")
axes[0].set_ylabel("%"); axes[0].tick_params(axis="x", rotation=30)
axes[0].legend(fontsize=8)
for i, v in enumerate(stress_df["pct_pricing_adequate"]):
    axes[0].text(i, v+1, f"{v:.0f}%", ha="center", fontsize=9)

axes[1].bar(stress_df["scenario"], stress_df["stressed_ni_cr"], color=ni_colors)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Net Income Under Stress (₹ Cr)")
axes[1].set_ylabel("₹ Crore"); axes[1].tick_params(axis="x", rotation=30)

axes[2].plot(stress_df["pd_multiplier"], stress_df["stressed_el_cr"],
              marker="o", color=PAL["danger"], linewidth=2, markersize=7, label="Stressed EL")
axes[2].plot(stress_df["pd_multiplier"], stress_df["stressed_ni_cr"],
              marker="s", color=PAL["primary"], linewidth=2, markersize=7, label="Net Income")
axes[2].axhline(0, color="black", linewidth=0.8)
axes[2].set_title("EL vs NI as PD Multiplier Increases")
axes[2].set_xlabel("PD Multiplier"); axes[2].set_ylabel("₹ Crore")
axes[2].legend(fontsize=9)

plt.tight_layout()
savefig("01_pricing_stress_test.png")
log.info("Pricing stress test complete.")


══════════════════════════════════════════════════════════════════════
  STEP 18 — PRICING ADEQUACY STRESS TESTING
══════════════════════════════════════════════════════════════════════

  Pricing Adequacy Under Stress:
       scenario  pd_multiplier  avg_stressed_pd_pct  stressed_el_cr  stressed_ni_cr  pct_pricing_adequate  pct_loans_negative_ni
       Baseline            1.0                57.65         1375.94          205.68                  52.0                   66.6
    Mild Stress            1.2                69.19         1453.41          128.21                  43.4                   68.9
Moderate Stress            1.5                84.63         1528.39           53.22                  40.8                   70.1
  Severe Stress            2.0                96.98         1577.28            4.34                  39.8                   70.3
 Extreme Stress            3.0                99.67         1586.79           -5.17                  39.6                   70.4
14:29

In [27]:
# =============================================================================
# CELL 17 — STEP 16: Executive Visual Analytics Dashboard
# =============================================================================

section("STEP 16 — EXECUTIVE PRICING ANALYTICS DASHBOARD")

fig = plt.figure(figsize=(22, 16))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.38)
fig.suptitle("Executive Pricing Intelligence Dashboard", fontsize=16,
             fontweight="bold", color=PAL["primary"])

gc = [GRADE_COLORS.get(str(g)[0], PAL["neutral"]) for g in grade_profit["risk_grade"]]

# ── P1: Avg rate by grade ─────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
bars = ax1.bar(grade_profit["risk_grade"], grade_profit["avg_rate"], color=gc)
ax1.axhline(price_df["recommended_rate"].mean(), color="black", linestyle="--",
             linewidth=1, label=f"Portfolio avg {price_df['recommended_rate'].mean():.1f}%")
ax1.set_title("Avg Interest Rate by Grade")
ax1.set_ylabel("Rate (%)"); ax1.legend(fontsize=8)
for bar, v in zip(bars, grade_profit["avg_rate"]):
    ax1.text(bar.get_x()+bar.get_width()/2, v+0.2, f"{v:.1f}%", ha="center", fontsize=9)

# ── P2: Rate distribution by grade ───────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
for g, color in GRADE_COLORS.items():
    sub = price_df[price_df["risk_grade"] == g]["recommended_rate"]
    if len(sub) > 5:
        ax2.hist(sub, bins=20, alpha=0.65, color=color, label=f"Grade {g}", density=True)
ax2.set_title("Rate Distribution by Grade")
ax2.set_xlabel("Interest Rate (%)"); ax2.set_ylabel("Density"); ax2.legend(fontsize=8)

# ── P3: Net income by grade ────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ni_c = [PAL["success"] if v > 0 else PAL["danger"] for v in grade_profit["avg_net_income"]]
ax3.bar(grade_profit["risk_grade"], grade_profit["avg_net_income"], color=ni_c)
ax3.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax3.set_title("Avg Net Income per Loan (₹)"); ax3.set_ylabel("Net Income (₹)")

# ── P4: RAROC by grade ────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
rc = [PAL["success"] if v > 1 else PAL["warning"] if v > 0 else PAL["danger"]
       for v in grade_profit["avg_raroc"]]
ax4.bar(grade_profit["risk_grade"], grade_profit["avg_raroc"], color=rc)
ax4.axhline(1.0, color=PAL["success"], linestyle="--", linewidth=1, label="RAROC target = 1.0")
ax4.set_title("Avg RAROC by Grade"); ax4.set_ylabel("RAROC"); ax4.legend(fontsize=8)

# ── P5: Risk premium vs PD scatter ───────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
samp = price_df.sample(min(3000, len(price_df)), random_state=SEED)
for g, color in GRADE_COLORS.items():
    sub = samp[samp["risk_grade"] == g]
    if len(sub):
        ax5.scatter(sub["pd_score"], sub["risk_premium_pct"],
                     s=5, alpha=0.35, color=color, label=f"Grade {g}")
ax5.set_title("Risk Premium vs PD Score")
ax5.set_xlabel("PD"); ax5.set_ylabel("Risk Premium (%)"); ax5.legend(fontsize=7, markerscale=4)

# ── P6: % Profitable by grade ─────────────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
ax6.bar(grade_profit["risk_grade"], grade_profit["pct_profitable"], color=gc)
ax6.set_title("% Profitable Loans by Grade")
ax6.set_ylabel("%"); ax6.set_ylim(0, 115)
for i, (v, cnt) in enumerate(zip(grade_profit["pct_profitable"], grade_profit["count"])):
    ax6.text(i, v+1, f"{v:.0f}%\n(n={int(cnt):,})", ha="center", fontsize=8)

# ── P7: Scenario comparison (full width) ──────────────────────────────────
ax7 = fig.add_subplot(gs[2, :])
x = np.arange(len(sim_df)); w = 0.35
c1 = [PAL["success"] if v >= 0 else PAL["danger"] for v in sim_df["ni_delta_cr"]]
c2 = [PAL["danger"]  if v >= 0 else PAL["success"] for v in sim_df["el_delta_cr"]]
ax7.bar(x - w/2, sim_df["ni_delta_cr"], w, color=c1, label="Net Income Δ vs Baseline (₹Cr)")
ax7.bar(x + w/2, sim_df["el_delta_cr"], w, color=c2, alpha=0.75, label="Expected Loss Δ vs Baseline (₹Cr)")
ax7.axhline(0, color="black", linewidth=0.8)
ax7.set_xticks(x)
ax7.set_xticklabels(sim_df["scenario"], rotation=20, ha="right", fontsize=9)
ax7.set_title("Scenario Impact vs Baseline — Net Income & Expected Loss (₹ Crore)")
ax7.set_ylabel("₹ Crore"); ax7.legend(fontsize=9)
for xi, v in zip(x - w/2, sim_df["ni_delta_cr"]):
    ax7.text(xi, v + (0.2 if v >= 0 else -0.5), f"{v:+.1f}", ha="center", fontsize=8)

plt.tight_layout()
savefig("02_executive_pricing_dashboard.png", dpi=120)
log.info("Executive dashboard saved.")


══════════════════════════════════════════════════════════════════════
  STEP 16 — EXECUTIVE PRICING ANALYTICS DASHBOARD
══════════════════════════════════════════════════════════════════════
14:29:20 | INFO     |   Figure: 02_executive_pricing_dashboard.png
14:29:20 | INFO     | Executive dashboard saved.


In [28]:
# =============================================================================
# CELL 18 — STEP 17: Advanced Visual Analytics
# =============================================================================

section("STEP 17 — ADVANCED VISUAL ANALYTICS")

# ── Figure: Risk-Return Matrix ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Risk–Return Pricing Matrix", fontsize=14, fontweight="bold", color=PAL["primary"])

samp = price_df.sample(min(4000, len(price_df)), random_state=SEED)
ni_clipped = samp["net_income"].clip(
    samp["net_income"].quantile(0.02),
    samp["net_income"].quantile(0.98)
)
sc = axes[0].scatter(
    samp["pd_score"], samp["recommended_rate"],
    c=ni_clipped, cmap="RdYlGn", s=6, alpha=0.45
)
plt.colorbar(sc, ax=axes[0], label="Net Income (₹)")
axes[0].set_title("PD vs Rate\n(colour = Net Income)")
axes[0].set_xlabel("Probability of Default")
axes[0].set_ylabel("Recommended Rate (%)")

# Efficient frontier by grade
for g, color in GRADE_COLORS.items():
    sub = price_df[price_df["risk_grade"] == g]
    if len(sub) > 5:
        bubble_size = sub["loan_amount"].sum() / 1e5
        axes[1].scatter(
            sub["pd_score"].mean(), sub["raroc"].mean(),
            s=max(bubble_size, 50), color=color, alpha=0.85,
            edgecolors="black", linewidth=0.5, label=f"Grade {g}"
        )
        axes[1].annotate(
            f" G-{g}", (sub["pd_score"].mean(), sub["raroc"].mean()),
            fontsize=9, fontweight="bold"
        )
axes[1].axhline(1.0, color="black", linestyle="--", linewidth=1, label="RAROC = 1.0")
axes[1].set_title("Risk–Return Efficient Frontier\n(bubble = total exposure)")
axes[1].set_xlabel("Avg PD (Risk)")
axes[1].set_ylabel("Avg RAROC (Return)")
axes[1].legend(fontsize=9)
plt.tight_layout()
savefig("03_risk_return_matrix.png")

# ── Figure: Profitability Waterfall ───────────────────────────────────────
waterfall_items = [
    ("Gross Interest",          price_df["gross_interest"].sum()),
    ("− Expected Loss",        -price_df["expected_loss"].sum()),
    ("− Servicing Cost",       -price_df["servicing_cost"].sum()),
    ("− Acquisition Cost",     -price_df["cac"].sum()),
    ("− Capital Cost",         -price_df["capital_cost"].sum()),
    ("= Net Income",            price_df["net_income"].sum()),
]
labels = [w[0] for w in waterfall_items]
values = [w[1] / 1e7 for w in waterfall_items]
wf_c   = [PAL["success"] if v >= 0 else PAL["danger"] for v in values]
wf_c[-1] = PAL["primary"]

fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.bar(labels, values, color=wf_c, edgecolor="white", linewidth=0.5)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Portfolio P&L Waterfall (₹ Crore)", fontsize=13, fontweight="bold")
ax.set_ylabel("₹ Crore"); ax.tick_params(axis="x", rotation=20)
for bar, v in zip(bars, values):
    ax.text(bar.get_x()+bar.get_width()/2,
             v + (0.2 if v >= 0 else -0.6),
             f"₹{v:.2f}Cr", ha="center", fontsize=9, fontweight="bold")
plt.tight_layout()
savefig("04_profitability_waterfall.png")

# ── Figure: Rate × Tenure Heatmap ─────────────────────────────────────────
price_df["tenure_bucket"] = pd.cut(
    price_df["tenure_months"].fillna(24),
    bins=[0, 12, 24, 36, 48, 120],
    labels=["0–12m", "13–24m", "25–36m", "37–48m", "49m+"]
)
if price_df["tenure_bucket"].nunique() > 1:
    hmap = price_df.groupby(
        ["risk_grade", "tenure_bucket"], observed=True
    )["recommended_rate"].mean().unstack(fill_value=np.nan)

    fig, ax = plt.subplots(figsize=(12, 6))
    cmap_hm = LinearSegmentedColormap.from_list("rate", ["#2E8B57", "#E8A838", "#D94F3D"])
    sns.heatmap(hmap, annot=True, fmt=".1f", cmap=cmap_hm, ax=ax,
                 linewidths=0.5, annot_kws={"size": 10},
                 cbar_kws={"label": "Avg Rate (%)"})
    ax.set_title("Interest Rate Heatmap — Grade × Tenure Bucket",
                  fontsize=13, fontweight="bold")
    ax.set_xlabel("Tenure Bucket"); ax.set_ylabel("Risk Grade")
    ax.tick_params(axis="y", rotation=0)
    plt.tight_layout()
    savefig("05_rate_tenure_heatmap.png")

# ── Figure: Pricing Sensitivity ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Portfolio Pricing Sensitivity", fontsize=13, fontweight="bold", color=PAL["primary"])

sc = [PAL["success"] if v >= 0 else PAL["danger"] for v in sens_df["rate_shift_pp"]]
axes[0].bar(sens_df["rate_shift_pp"].astype(str), sens_df["total_ni_cr"], color=sc)
axes[0].set_title("Net Income vs Rate Shift (₹ Cr)")
axes[0].set_xlabel("Rate Shift (pp)"); axes[0].set_ylabel("Net Income (₹ Cr)")
for i, v in enumerate(sens_df["total_ni_cr"]):
    axes[0].text(i, v + 0.2, f"₹{v:.1f}Cr", ha="center", fontsize=8)

axes[1].plot(sens_df["rate_shift_pp"], sens_df["pct_affordable"],
              marker="o", color=PAL["success"], linewidth=2, label="% Affordable")
axes[1].plot(sens_df["rate_shift_pp"], sens_df["pct_profitable"],
              marker="s", color=PAL["primary"], linewidth=2, label="% Profitable")
axes[1].set_title("Affordability & Profitability vs Rate Shift")
axes[1].set_xlabel("Rate Shift (pp)"); axes[1].set_ylabel("%")
axes[1].legend(fontsize=9)
plt.tight_layout()
savefig("06_pricing_sensitivity.png")

# ── Figure: Behavioral modifier distribution ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Behavioral Pricing Modifier Analysis", fontsize=13,
             fontweight="bold", color=PAL["primary"])

axes[0].hist(price_df["behavioral_mod"], bins=40, color=PAL["highlight"], alpha=0.8, edgecolor="white")
axes[0].axvline(0, color="black", linestyle="--", linewidth=1.5)
axes[0].axvline(price_df["behavioral_mod"].mean(), color=PAL["primary"], linestyle="--",
                 linewidth=1.5, label=f"Mean={price_df['behavioral_mod'].mean():+.2f}%")
axes[0].set_title("Behavioral Modifier Distribution")
axes[0].set_xlabel("Behavioral Modifier (%)"); axes[0].legend(fontsize=9)

if "acquisition_channel" in price_df.columns:
    chan = price_df.groupby("acquisition_channel")["raroc"].mean().sort_values()
    chan_c = [PAL["success"] if v > 1 else PAL["warning"] if v > 0 else PAL["danger"]
               for v in chan.values]
    axes[1].barh(chan.index, chan.values, color=chan_c)
    axes[1].axvline(1.0, color=PAL["primary"], linestyle="--", linewidth=1.5, label="RAROC=1.0")
    axes[1].set_title("Avg RAROC by Acquisition Channel")
    axes[1].set_xlabel("RAROC"); axes[1].legend(fontsize=9)

plt.tight_layout()
savefig("07_behavioral_modifier.png")

log.info("Advanced visual analytics complete.")
print(f"\n  ✅  {len(list(Path(DASH_DIR).glob('*.png')))} figures saved to {DASH_DIR}")


══════════════════════════════════════════════════════════════════════
  STEP 17 — ADVANCED VISUAL ANALYTICS
══════════════════════════════════════════════════════════════════════
14:29:34 | INFO     |   Figure: 03_risk_return_matrix.png
14:29:35 | INFO     |   Figure: 04_profitability_waterfall.png
14:29:35 | INFO     |   Figure: 05_rate_tenure_heatmap.png
14:29:35 | INFO     | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
14:29:35 | INFO     | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
14:29:35 | INFO     |   Figure: 06_pricing_sensitivity.png
14:29:36 | INFO     |   Figure: 07_behavioral_modifier.png
14:29:36 | INFO     | Advanced visual analytics complete.

  ✅  8 figures saved to C:\Users\abhir

In [29]:
# =============================================================================
# CELL 19 — STEP 19: Production Pricing Pipeline
# =============================================================================

section("STEP 19 — PRODUCTION PRICING PIPELINE")

# ── Persist configs ────────────────────────────────────────────────────────
for fname, obj in [
    ("pricing_config.json",          PRICING_CONFIG),
    ("pricing_policy.json",          PRICING_POLICY),
    ("segment_overrides.json",       SEGMENT_PRICING_OVERRIDE),
    ("cac_by_channel.json",          CAC_BY_CHANNEL),
]:
    with open(os.path.join(PIP_DIR, fname), "w") as f:
        json.dump(obj, f, indent=2)

PRODUCTION_CODE = '''
# ================================================================
# PRODUCTION PRICING PIPELINE — Module 6
# Place in: iitg/pricing_engine/pipelines/pricing_pipeline.py
# Usage   : from pricing_pipeline import price_borrower, score_batch
# ================================================================
import json
import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR = r"C:\\Users\\abhir\\OneDrive\\Desktop\\iitg"
PIP_DIR  = f"{BASE_DIR}/pricing_engine/pipelines"

_cfg     = json.load(open(f"{PIP_DIR}/pricing_config.json"))
_policy  = json.load(open(f"{PIP_DIR}/pricing_policy.json"))
_seg_ovr = json.load(open(f"{PIP_DIR}/segment_overrides.json"))
_cac     = json.load(open(f"{PIP_DIR}/cac_by_channel.json"))

SERVICING_PA  = 500
CAPITAL_RATIO = 0.08


def _risk_premium(pd_val, lgd, tenure_months):
    pd_val = float(np.clip(pd_val, 0.001, 0.999))
    yrs    = max(tenure_months / 12, 0.25)
    cum_pd = 1 - (1 - pd_val) ** yrs
    return cum_pd * lgd / yrs * 100


def _beh_mod(sv=0.3, pr=0.8, ae=10, fs=0.3):
    s  = (sv - 0.30) * 4.0
    s -= (pr - 0.80) * 3.0
    s -= (min(ae, 20) / 20) * 1.0
    s += (fs - 0.30) * 3.0
    return float(np.clip(s, -2.0, 3.0))


def price_borrower(
    pd_score, risk_grade, loan_amount, tenure_months,
    annual_income=None, spending_volatility=0.3,
    payment_regularity=0.8, app_engagement=10,
    financial_stress=0.3, ml_persona=None,
    acquisition_channel=None,
) -> dict:
    """
    Single-borrower pricing. Returns rate, EMI, limit, NI, RAROC, decision.
    """
    g   = str(risk_grade)[0].upper()
    g   = g if g in _policy else "C"
    pol = _policy[g]
    lgd = pol["lgd"]

    rp   = _risk_premium(pd_score, lgd, tenure_months)
    bm   = _beh_mod(spending_volatility, payment_regularity, app_engagement, financial_stress)
    bm   = float(np.clip(bm, -2.0, pol["behavioral_mod_cap"]))
    sa   = _seg_ovr.get(ml_persona, {}).get("rate_adj", 0.0) if ml_persona else 0.0
    raw  = _cfg["base_rate"] + _cfg["operational_spread"] + rp + _cfg["target_margin"] + bm + sa
    rate = float(np.clip(raw, pol["rate_floor"], pol["rate_ceil"]))

    mr   = rate / 100 / 12
    n    = int(tenure_months)
    emi  = loan_amount * mr * (1+mr)**n / ((1+mr)**n - 1) if mr > 0 else loan_amount / n
    mi   = annual_income / 12 if annual_income else None
    er   = emi / mi if mi else None
    aff  = (er < _cfg["emi_burden_ceiling"]) if er else True

    yrs       = n / 12
    cum_pd    = 1 - (1 - pd_score) ** yrs
    el        = cum_pd * lgd * loan_amount
    gross     = emi * n - loan_amount
    cac_val   = _cac.get(acquisition_channel, 1000)
    svc       = SERVICING_PA * yrs
    cap_cost  = loan_amount * CAPITAL_RATIO * yrs * (rate / 100)
    ni        = gross - el - svc - cac_val - cap_cost
    raroc     = ni / max(el * 12.5 * 0.08, 1)

    # Decision
    if g == "E" or pd_score > 0.55 or not aff:
        decision = "DECLINE"
    elif g in ["A","B"] and ni > 0 and raroc > 0.5:
        decision = "APPROVE"
    elif g == "C" and ni > 0:
        decision = "CONDITIONAL"
    else:
        decision = "MANUAL_REVIEW"

    return {
        "risk_grade":       g,
        "recommended_rate": round(rate, 2),
        "risk_premium_pct": round(rp, 3),
        "behavioral_mod":   round(bm, 2),
        "segment_adj":      round(sa, 2),
        "emi":              round(emi, 2),
        "emi_to_income":    round(er, 4) if er else None,
        "affordability_ok": aff,
        "net_income":       round(ni, 2),
        "raroc":            round(raroc, 4),
        "approval":         pol["approval"],
        "decision":         decision,
    }


def score_batch(df: pd.DataFrame) -> pd.DataFrame:
    """Score a full DataFrame of borrowers."""
    return pd.DataFrame([
        price_borrower(
            pd_score           = row.get("pd_score", 0.15),
            risk_grade         = row.get("risk_grade", "C"),
            loan_amount        = row.get("loan_amount", 100000),
            tenure_months      = row.get("loan_tenure_months", 24),
            annual_income      = row.get("annual_income"),
            spending_volatility= row.get("spending_volatility_index", 0.3),
            payment_regularity = row.get("payment_regularization_score", 0.8),
            app_engagement     = row.get("app_usage_mean", 10),
            financial_stress   = row.get("financial_stress_index", 0.3),
            ml_persona         = row.get("ml_persona"),
            acquisition_channel= row.get("acquisition_channel"),
        )
        for _, row in df.iterrows()
    ])


if __name__ == "__main__":
    # Quick smoke test
    result = price_borrower(
        pd_score=0.06, risk_grade="B", loan_amount=200000,
        tenure_months=24, annual_income=600000,
        spending_volatility=0.2, payment_regularity=0.9
    )
    print(result)
'''

pipeline_path = os.path.join(PIP_DIR, "pricing_pipeline.py")
with open(pipeline_path, "w", encoding="utf-8") as f:
    f.write(PRODUCTION_CODE)

# ── Registry ──────────────────────────────────────────────────────────────
registry = {
    "module":              "Module 6 — Dynamic Risk-Based Pricing",
    "created_at":          datetime.now().isoformat(),
    "total_loans_priced":  int(len(price_df)),
    "avg_rate":            round(price_df["recommended_rate"].mean(), 3),
    "avg_raroc":           round(price_df["raroc"].mean(), 4),
    "pct_profitable":      round(price_df["profitable"].mean() * 100, 1),
    "total_net_income_cr": round(price_df["net_income"].sum() / 1e7, 2),
    "scenarios_simulated": len(SCENARIOS),
    "stress_scenarios":    len(STRESS_SCENARIOS),
    "figures_generated":   len(list(Path(DASH_DIR).glob("*.png"))),
    "output_dirs": {
        "dashboards":    DASH_DIR,
        "reports":       RPT_DIR,
        "simulations":   SIM_DIR,
        "explainability":EXP_DIR,
        "pipelines":     PIP_DIR,
        "policies":      POL_DIR,
    },
}
with open(os.path.join(PE_DIR, "module6_registry.json"), "w") as f:
    json.dump(registry, f, indent=2)

log.info("Production pipeline and registry saved.")
print(f"  ✅  Production pipeline: {pipeline_path}")
print(f"  ✅  Module registry saved to pricing_engine/module6_registry.json")


══════════════════════════════════════════════════════════════════════
  STEP 19 — PRODUCTION PRICING PIPELINE
══════════════════════════════════════════════════════════════════════
14:29:44 | INFO     | Production pipeline and registry saved.
  ✅  Production pipeline: C:\Users\abhir\OneDrive\Desktop\iitg\pricing_engine\pipelines\pricing_pipeline.py
  ✅  Module registry saved to pricing_engine/module6_registry.json


In [30]:
# =============================================================================
# CELL 20 — Executive Pricing Recommendations Report
# =============================================================================

section("EXECUTIVE PRICING RECOMMENDATIONS")

best_grade  = grade_profit.sort_values("avg_raroc", ascending=False).iloc[0]
worst_grade = grade_profit.sort_values("avg_raroc").iloc[0]
cons_ni = sim_df.loc[sim_df["scenario"]=="Conservative Lending", "total_net_income_cr"].values[0]
aggr_ni = sim_df.loc[sim_df["scenario"]=="Aggressive Growth",   "total_net_income_cr"].values[0]
base_ni = sim_df.loc[sim_df["scenario"]=="Baseline",             "total_net_income_cr"].values[0]
sev_adq = stress_df.loc[stress_df["scenario"]=="Severe Stress",  "pct_pricing_adequate"].values[0]

EXEC_REPORT = f"""
╔══════════════════════════════════════════════════════════════════════╗
║  MODULE 6 — EXECUTIVE PRICING INTELLIGENCE REPORT                    ║
║  Prepared for : CRO / Chief Pricing Officer / Portfolio Head          ║
║  Generated    : {datetime.now().strftime('%Y-%m-%d %H:%M')}                              ║
╠══════════════════════════════════════════════════════════════════════╣
║  PORTFOLIO PRICING KPIs                                              ║
║  Avg Interest Rate     : {price_df['recommended_rate'].mean():.2f}%                         ║
║  Avg RAROC             : {price_df['raroc'].mean():.4f}                         ║
║  Avg NIM               : {price_df['nim_pct'].mean():.2f}%                           ║
║  % Profitable Loans    : {price_df['profitable'].mean()*100:.1f}%                          ║
║  Total Net Income      : ₹{price_df['net_income'].sum()/1e7:.2f} Crore                   ║
╠══════════════════════════════════════════════════════════════════════╣
║  STRATEGIC FINDINGS                                                  ║
║                                                                      ║
║  1. Grade {best_grade['risk_grade']} delivers highest RAROC ({best_grade['avg_raroc']:.2f}) — scaling  ║
║     Grade A/B acquisition is the highest capital-yield strategy.     ║
║                                                                      ║
║  2. Grade {worst_grade['risk_grade']} shows RAROC {worst_grade['avg_raroc']:.2f} — current pricing    ║
║     is inadequate for the risk level. Consider rate increase or exit.║
║                                                                      ║
║  3. Conservative strategy generates ₹{cons_ni:.2f}Cr NI vs Baseline ║
║     ₹{base_ni:.2f}Cr — quality improvement outweighs volume loss.   ║
║                                                                      ║
║  4. Aggressive growth (−2% rate cut) reduces NI to ₹{aggr_ni:.2f}Cr ║
║     — unfavourable risk-return tradeoff; not recommended.            ║
║                                                                      ║
║  5. Behavioral modifier is effective: stable borrowers earn up to    ║
║     −2% discount; volatile borrowers carry +3% premium.             ║
║                                                                      ║
║  6. Severe stress scenario: only {sev_adq:.0f}% of loans priced       ║
║     adequately. Recommend +2pp stress buffer on grades C/D.          ║
║                                                                      ║
║  7. DSA/Branch channel loans show lowest RAROC — CAC overhang.       ║
║     Re-price DSA channel +1.5pp or reduce origination allocation.    ║
╠══════════════════════════════════════════════════════════════════════╣
║  RECOMMENDED ACTIONS                                                 ║
║  • Deploy Balanced strategy as default pricing mode                  ║
║  • Reduce Grade E exposure to < 5% of portfolio book                ║
║  • Activate behavioral modifier in real-time scoring pipeline        ║
║  • Apply segment override for High-Value Loyal borrowers             ║
║  • Monitor RAROC monthly; alert if Grade B average drops < 0.80     ║
║  • Add +2pp stress buffer to Grade C/D floors as EL reserve          ║
║  • Implement channel-adjusted pricing for DSA/Branch originations    ║
║  • Review pricing at 6-month cycle using PSI monitoring (Module 5)   ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(EXEC_REPORT)
with open(os.path.join(RPT_DIR, "EXECUTIVE_PRICING_REPORT.txt"), "w", encoding="utf-8") as f:
    f.write(EXEC_REPORT)
log.info("Executive pricing report saved.")


══════════════════════════════════════════════════════════════════════
  EXECUTIVE PRICING RECOMMENDATIONS
══════════════════════════════════════════════════════════════════════

╔══════════════════════════════════════════════════════════════════════╗
║  MODULE 6 — EXECUTIVE PRICING INTELLIGENCE REPORT                    ║
║  Prepared for : CRO / Chief Pricing Officer / Portfolio Head          ║
║  Generated    : 2026-05-22 14:30                              ║
╠══════════════════════════════════════════════════════════════════════╣
║  PORTFOLIO PRICING KPIs                                              ║
║  Avg Interest Rate     : 34.21%                         ║
║  Avg RAROC             : -0.4741                         ║
║  Avg NIM               : -7.55%                           ║
║  % Profitable Loans    : 23.9%                          ║
║  Total Net Income      : ₹-0.04 Crore                   ║
╠══════════════════════════════════════════════════════════════════════╣
║  STRATEGIC

In [31]:
# =============================================================================
# CELL 21 — Final Summary & Module Orchestrator
# =============================================================================

print("\n" + "═" * 70)
print("  MODULE 6 — PART 2 COMPLETE  |  FULL MODULE COMPLETE")
print("═" * 70)
print(f"  Loans priced            : {len(price_df):,}")
print(f"  Avg recommended rate    : {price_df['recommended_rate'].mean():.2f}%")
print(f"  Avg RAROC               : {price_df['raroc'].mean():.4f}")
print(f"  Avg NIM                 : {price_df['nim_pct'].mean():.2f}%")
print(f"  % Profitable loans      : {price_df['profitable'].mean()*100:.1f}%")
print(f"  Total net income        : ₹{price_df['net_income'].sum()/1e7:.2f} Cr")
print(f"  Scenarios simulated     : {len(SCENARIOS)}")
print(f"  Stress scenarios        : {len(STRESS_SCENARIOS)}")
print(f"  Figures generated       : {len(list(Path(DASH_DIR).glob('*.png')))}")
print(f"\n  Output directories:")
print(f"  📊 Dashboards    → {DASH_DIR}")
print(f"  📋 Reports       → {RPT_DIR}")
print(f"  🎲 Simulations   → {SIM_DIR}")
print(f"  💡 Explainability → {EXP_DIR}")
print(f"  🔧 Pipelines     → {PIP_DIR}")
print(f"  📜 Policies      → {POL_DIR}")
print("═" * 70)
log.info("Module 6 complete.")


def get_module6_artefacts() -> dict:
    """Export all Module 6 outputs for downstream modules (7–15)."""
    return {
        # Core DataFrames
        "price_df":          price_df,
        "grade_profit":      grade_profit,
        "sim_df":            sim_df,
        "stress_df":         stress_df,
        "sens_df":           sens_df,
        "grade_exposure":    grade_exposure,
        "seg_pricing":       seg_pricing,
        "fairness_summary":  fairness_summary,
        # Configs
        "PRICING_CONFIG":    PRICING_CONFIG,
        "PRICING_POLICY":    PRICING_POLICY,
        "SEGMENT_PRICING_OVERRIDE": SEGMENT_PRICING_OVERRIDE,
        "CAC_BY_CHANNEL":    CAC_BY_CHANNEL,
        "SCENARIOS":         SCENARIOS,
        "STRESS_SCENARIOS":  STRESS_SCENARIOS,
        # Functions
        "price_loan":        price_loan,
        "optimize_tenure":   optimize_tenure,
        "optimize_credit_limit": optimize_credit_limit,
        "compute_full_profitability": compute_full_profitability,
        "integrated_underwriting_decision": integrated_underwriting_decision,
        "explain_pricing_decision": explain_pricing_decision,
        "compute_risk_premium":     compute_risk_premium,
        "compute_behavioral_modifier": compute_behavioral_modifier,
        # Paths
        "PE_DIR":   PE_DIR,   "RPT_DIR":  RPT_DIR,
        "DASH_DIR": DASH_DIR, "SIM_DIR":  SIM_DIR,
        "EXP_DIR":  EXP_DIR,  "PIP_DIR":  PIP_DIR,
        "POL_DIR":  POL_DIR,
    }

# artefacts = get_module6_artefacts()  # uncomment to import in Module 7+
print("\n✅  Module 6 orchestrator ready. Call get_module6_artefacts() to export.")


══════════════════════════════════════════════════════════════════════
  MODULE 6 — PART 2 COMPLETE  |  FULL MODULE COMPLETE
══════════════════════════════════════════════════════════════════════
  Loans priced            : 65,000
  Avg recommended rate    : 34.21%
  Avg RAROC               : -0.4741
  Avg NIM                 : -7.55%
  % Profitable loans      : 23.9%
  Total net income        : ₹-0.04 Cr
  Scenarios simulated     : 7
  Stress scenarios        : 5
  Figures generated       : 8

  Output directories:
  📊 Dashboards    → C:\Users\abhir\OneDrive\Desktop\iitg\pricing_engine\dashboards
  📋 Reports       → C:\Users\abhir\OneDrive\Desktop\iitg\pricing_engine\reports
  🎲 Simulations   → C:\Users\abhir\OneDrive\Desktop\iitg\pricing_engine\simulations
  💡 Explainability → C:\Users\abhir\OneDrive\Desktop\iitg\pricing_engine\explainability
  🔧 Pipelines     → C:\Users\abhir\OneDrive\Desktop\iitg\pricing_engine\pipelines
  📜 Policies      → C:\Users\abhir\OneDrive\Desktop\iitg\pri